# Basketball Scouting Analysis - Modular Pipeline

A comprehensive, modular pipeline for analyzing international basketball players and identifying NBA prospects for the Sacramento Kings.

---

## 📁 Project Structure

```
modular/
├── README.md                      # This file
├── config.py                      # Configuration with column schemas
├── utils.py                       # Shared utility functions
├── main.py                        # Pipeline orchestration
│
├── data/                          # Data loading & validation
│   ├── __init__.py
│   └── loader.py                  # JSON loading, validation, collision detection
│
├── database/                      # Database operations
│   ├── __init__.py
│   └── manager.py                 # SQLite schema, ETL, sanity checks
│
├── features/                      # Feature engineering
│   ├── __init__.py
│   └── statistics.py              # Per-game, per-36, efficiency metrics
│
├── analysis/                      # Performance analysis
│   ├── __init__.py
│   └── performance.py             # Trajectories, trends, league comparisons
│
├── models/                        # Machine learning
│   ├── __init__.py
│   └── predictor.py               # NBA success prediction model
│
├── scouting/                      # Scouting logic
│   ├── __init__.py
│   └── targets.py                 # Team weights, prospect scoring & ranking
│
└── reporting/                     # Reporting & visualization
    ├── __init__.py
    └── generator.py               # EDA plots, reports, outputs
```

---

## 📊 Module Descriptions

### 1. **config.py** - Configuration Management
**Purpose**: Centralized configuration and column schemas for the entire pipeline.

**Key Components**:
- `Config` class: All parameters (file paths, thresholds, ML hyperparameters)
- `ColumnSchemas` class: Column definitions by type and purpose

**Column Schema Organization**:
- **ID Columns**: `player_id`, `first_name`, `last_name`, `team`, `league`
- **Numerical Discrete**: Counts like `games`, `assists`, `steals`
- **Numerical Continuous**: Measurements like `minutes`, `points`
- **Percentages**: Rates like `true_shooting_percentage`, `usage_percentage`
- **Categorical**: `league`, `team`, `season_type`
- **Target Variables**: `nba_success`, `scout_score`, `nba_success_prob`
- **Feature Groups**: Organized by ML usage (basic, advanced, trajectory)

**Why This Structure**:
- Enables automated preprocessing based on data type
- Facilitates focused EDA on relevant column groups
- Makes feature engineering systematic and reproducible
- Single source of truth for all configuration values

### 2. **utils.py** - Shared Utilities
**Purpose**: Common functions used across multiple modules.

**Key Functions**:
- `safe_div()`: Division with NaN handling (prevents errors)
- `normalize_metric()`: Scale metrics to 0-100 for comparison
- `format_stat()` / `format_probability_range()`: Display formatting
- `calculate_age_bonus()` / `calculate_improvement_bonus()`: Multipliers
- `calculate_efg_percentage()`: Effective field goal percentage

**Design Principle**: DRY (Don't Repeat Yourself) - define once, use everywhere.

### 3. **data/** - Data Loading & Validation
**Module**: `data/loader.py`

**Responsibilities**:
- Load JSON files into pandas DataFrames
- Process demographics (birth dates → ages)
- Create player IDs (handle name collisions with birth year)
- Comprehensive validation:
  - Check required columns
  - Validate numeric ranges (games ≤ 100, minutes ≤ 4500)
  - Validate percentages (0-1 scale)
  - Validate shooting logic (made ≤ attempted)
  - Detect records with zero stats
- Track data quality issues

**Key Class**: `DataLoader`

**Why This Approach**:
- Catch data issues early before they propagate
- Maintain referential integrity (no orphaned stats)
- Document all quality issues for transparency
- Name collision resolution prevents silent errors

### 4. **database/** - Database Operations
**Module**: `database/manager.py`

**Responsibilities**:
- Create SQLite database with optimized pragmas (WAL, foreign keys)
- Define schema with constraints (CHECK, FOREIGN KEY)
- Create indexes for query performance
- ETL pipeline: filter orphans → load tables → log issues
- Run sanity checks (data completeness, top scorers)

**Key Class**: `DatabaseManager`

**Schema Design**:
- `players`: Demographics with age constraints
- `nba_stats`: NBA statistics with validation
- `intl_stats`: International stats with league dimension
- `data_quality_log`: Audit trail of issues

**Why SQLite**:
- Embedded database (no server needed)
- ACID compliance for data integrity
- Excellent query performance with proper indexes
- Portable single-file format

### 5. **features/** - Feature Engineering
**Module**: `features/statistics.py`

**Responsibilities**:
- Calculate per-game stats (PPG, APG, RPG, etc.)
- Calculate shooting percentages (FG%, 3P%, FT%, eFG%)
- Calculate efficiency rating
- Calculate per-36-minute stats
- Estimate usage percentage (if not provided)
- Calculate plus/minus per game

**Key Class**: `StatisticsCalculator`

**Why Separate Calculation Types**:
- Each method has single responsibility
- Easy to test individual calculations
- Simple to add new metrics
- Consistent use of `safe_div()` for robustness

### 6. **analysis/** - Performance Analysis
**Module**: `analysis/performance.py`

**Responsibilities**:
- Calculate player trajectories (first season → last season changes)
- Identify trending players (strong improvement patterns)
- Analyze league performance distributions
- Compare NBA vs International competition levels

**Key Class**: `PerformanceAnalyzer`

**Trajectory Metrics**:
- PPG change (scoring improvement)
- Efficiency change (overall game improvement)
- Seasons played (experience/consistency)

**Why Trajectory Analysis**:
- Improvement trajectory indicates adaptability and work ethic
- More predictive than single-season performance
- Identifies late bloomers who might be overlooked

### 7. **models/** - Machine Learning
**Module**: `models/predictor.py` (to be implemented)

**Planned Responsibilities**:
- Define NBA success criteria (MPG ≥ 15 and scoring/efficiency thresholds)
- Extract features from last international season
- Train calibrated Gradient Boosting classifier
- Generate diagnostic plots (calibration, ROC, PR curves)
- Calculate feature importance (Gini + permutation)
- Predict success probability for current prospects

**Why Gradient Boosting**:
- Handles non-linear relationships well
- Feature importance built-in
- Good performance with moderate sample sizes
- Calibration improves probability estimates

**Why Calibration**:
- Raw model probabilities can be overconfident
- Calibrated probabilities better represent true likelihood
- Critical for decision-making (scouting investments)

### 8. **scouting/** - Scouting Logic
**Module**: `scouting/targets.py` (to be implemented)

**Planned Responsibilities**:
- Calculate team weights based on successful NBA players
- Filter eligible prospects (2021, min games/MPG, age < 30)
- Calculate performance scores (normalized metrics)
- Apply multipliers:
  - Age bonus (younger = higher upside)
  - Improvement bonus (strong trajectory = higher potential)
  - Fit multiplier (match team needs)
  - ML multiplier (predicted success probability)
- Rank prospects by composite scout score

**Scout Score Formula**:
```
scout_score = performance_score × age_bonus × improvement_bonus × 
              fit_multiplier × ml_multiplier
```

**Why This Approach**:
- Balances current performance with potential
- Incorporates team-specific needs
- Uses ML prediction as one factor, not the only factor
- Transparent, explainable scoring system

### 9. **reporting/** - Report Generation
**Module**: `reporting/generator.py` (to be implemented)

**Planned Responsibilities**:
- Generate EDA visualizations (9-panel  plot)
- Generate ML diagnostic plots (calibration, ROC, PR)
- Print formatted prospect summaries (top 20)
- Export CSV with scouting recommendations
- Export JSON with ML metrics
- Generate data quality report

**Visualization Strategy**:
- EDA: Show data quality + distribution insights
- ML: Show model performance + reliability
- Reports: Actionable recommendations for scouts

---

## 🔬 Data Preprocessing Rationale

### Why We Preprocess This Way

#### 1. **Handling Missing Data**
**Approach**: Set invalid values to NaN rather than imputing

**Rationale**:
- Preserves data integrity (no false information)
- NaN propagates through calculations naturally
- Downstream methods handle NaN appropriately
- Imputation could introduce bias in small samples

**Example**: Player with 0 games → all per-game stats = NaN (correct)

#### 2. **Validation Thresholds**
**Approach**: Set extreme values to NaN (games > 100, minutes > 4500)

**Rationale**:
- Catches data entry errors (e.g., typos)
- No NBA/international player exceeds these thresholds
- More conservative than dropping entire records
- Documented in quality log for review

#### 3. **Name Collision Resolution**
**Approach**: Enhance player_id with birth year when duplicates exist

**Rationale**:
- Prevents merging different players with same name
- Birth year is stable identifier
- Maintains backward compatibility (simple names when no collision)
- Explicit logging of all collisions

#### 4. **Safe Division Everywhere**
**Approach**: Custom `safe_div()` function returns NaN for division by zero

**Rationale**:
- Prevents crashes from edge cases
- Mathematically correct (undefined → NaN)
- Consistent behavior across all calculations
- Easier to debug (NaN propagates, doesn't hide)

#### 5. **Vectorized Operations**
**Approach**: Use pandas vectorized operations instead of loops

**Rationale**:
- 10-100x faster on large datasets
- More readable code
- Less prone to errors
- Better memory efficiency

**Example**:
```python
# Slow (loop)
for i in range(len(df)):
    df.loc[i, 'ppg'] = df.loc[i, 'points'] / df.loc[i, 'games']

# Fast (vectorized)
df['ppg'] = safe_div(df['points'], df['games'])
```

#### 6. **Column Schema Approach**
**Approach**: Define column types explicitly in config

**Benefits**:
- Automated type-appropriate preprocessing
- Targeted EDA (e.g., correlation only on numerical)
- Consistent feature engineering
- Self-documenting code

**Preprocessing by Type**:
- **Numerical Discrete**: Check for negative values, extreme outliers
- **Numerical Continuous**: Check for negative values, distribution checks
- **Percentages**: Validate 0-1 range, check for impossible values
- **Categorical**: Check for unexpected categories, handle missing
- **ID Columns**: Ensure uniqueness, referential integrity

---

## 🤖 Modeling Approach

### Why This Simple Yet Effective Strategy

#### Model Selection: Gradient Boosting Classifier

**Why Not Deep Learning**:
- Limited training data (~50-200 examples)
- Tabular data (not images/text/sequences)
- Need interpretability for stakeholder trust
- Deep learning excels with 10,000+ examples

**Why Not Linear Models**:
- Non-linear relationships (e.g., age has optimal range)
- Feature interactions matter (young + improving ≠ same as old + improving)
- Need flexibility without manual feature engineering

**Why Gradient Boosting**:
- ✅ Excellent performance on tabular data
- ✅ Handles non-linearities and interactions
- ✅ Built-in feature importance
- ✅ Robust to outliers
- ✅ Works well with 50-500 examples
- ✅ Fast to train and predict

#### Key Modeling Decisions

**1. Binary Classification**
- **Target**: NBA success (Y/N) based on MPG ≥ 15 + scoring/efficiency
- **Why**: Clear, actionable decision for scouts
- **Alternative**: Regression on MPG (less interpretable)

**2. Calibration**
- **Method**: Platt scaling (sigmoid calibration)
- **Why**: Raw probabilities often overconfident
- **Impact**: Probabilities better match actual success rates
- **Use**: More reliable for decision-making

**3. Feature Engineering**
- **Basic**: Last season stats (PPG, APG, RPG, shooting %)
- **Advanced**: Efficiency metrics (TS%, usage%, eFG%)
- **Trajectory**: Career trends (PPG change, seasons played)
- **Why Trajectory**: Improvement pattern predicts adaptation

**4. Cross-Validation**
- **Method**: 5-fold stratified CV
- **Why**: More reliable performance estimate
- **Small data**: Critical for avoiding overfitting

**5. Evaluation Metrics**
- **Primary**: PR-AUC (precision-recall area under curve)
- **Why**: Class imbalance (few successes), care more about positives
- **Secondary**: ROC-AUC, Brier score, Precision@K
- **Why PR-AUC**: Better for imbalanced data than accuracy

**6. Feature Importance**
- **Methods**: Gini importance + permutation importance
- **Why Both**: Gini fast but biased; permutation unbiased but slow
- **Use**: Understanding which factors matter most

**7. Hyperparameter Choices**
```python
n_estimators=100      # More trees = better, 100 is sweet spot
max_depth=4           # Prevents overfitting on small data
learning_rate=0.1     # Standard value, balances speed and accuracy
```

**8. Train/Test Split**
- **Split**: 75% train, 25% test
- **Stratified**: Preserve class balance
- **Why**: Standard split for 50-200 samples

#### Model Limitations & Mitigations

**Limitation 1**: Small sample size

- **Impact**: High variance in estimates
- **Mitigation**: Cross-validation, bootstrapping for confidence intervals
- **Mitigation**: Conservative predictions (don't oversell certainty)

**Limitation 2**: Limited features

- **Impact**: May miss important factors (personality, injuries, etc.)
- **Mitigation**: Combine ML with expert judgment
- **Mitigation**: Use ML as one input to scout score, not only input

**Limitation 3**: Historical data only

- **Impact**: Game evolving (3-point shooting emphasis)
- **Mitigation**: Focus on recent training data
- **Mitigation**: Regularly retrain model

**Limitation 4**: Survivorship bias

- **Impact**: Only players who got NBA chance in training data
- **Mitigation**: Acknowledge in documentation
- **Mitigation**: Conservative probability interpretation

---

## 📈 Results

### Data Quality Assessment

**Total Records Processed**:
- Players: 20 unique individuals
- NBA Player-Seasons: [varies by dataset]
- International Player-Seasons: [varies by dataset]

**Data Quality Issues Identified**: [varies by dataset]
- Duplicate names resolved with birth year enhancement
- Extreme values flagged and set to NaN
- Orphaned records filtered (stats without demographics)
- Shooting logic validated (made ≤ attempted)

**Data Completeness**:
- NBA Data: [X]% complete across key fields
- International Data: [Y]% complete across key fields
- Completeness improved over time (better data in recent seasons)

### Statistical Analysis Results

**Player Trajectories** (International 2+ Seasons):
- Players analyzed: [varies]
- Average PPG improvement: +X.X points
- Average efficiency improvement: +Y.Y
- Top improvers: [list of players with strong trends]

**League Performance** (2021 Season):
- EuroLeague: Highest average PPG (competitive league)
- ACB: Balanced statistics, good talent pipeline
- [Other leagues]: [characteristics]

**NBA vs International Comparison**:
- NBA average PPG: ~X.X higher (as expected)
- International 3P%: Comparable to NBA (modern game)
- Efficiency: NBA higher due to better teammates

### Machine Learning Model Performance

#### Overall Metrics
- **ROC-AUC**: 0.XXX (0.5 = random, 1.0 = perfect)
  - Interpretation: [Good/Excellent] discrimination ability
- **PR-AUC**: 0.XXX (baseline = success rate)
  - Interpretation: [X]x better than random
- **Brier Score (Calibrated)**: 0.XXX (0 = perfect, 0.25 = random)
  - Interpretation: Well-calibrated probabilities

#### Cross-Validation Results
- **5-Fold CV ROC-AUC**: 0.XXX ± 0.XX
- **5-Fold CV PR-AUC**: 0.XXX ± 0.XX
- Interpretation: Stable performance, low variance

#### Precision at Top-K
- **Precision@10**: X%  - Of top 10 predicted, X succeed
- **Precision@20**: Y%  - Of top 20 predicted, Y succeed
- **Precision@30**: Z%  - Of top 30 predicted, Z succeed

#### Feature Importance (Top 5)
1. **[Feature Name]**: Importance score = X.XX
   - Interpretation: [Why this matters]
2. **[Feature Name]**: Importance score = X.XX
3. **[Feature Name]**: Importance score = X.XX
4. **[Feature Name]**: Importance score = X.XX
5. **[Feature Name]**: Importance score = X.XX

**Key Insight**: [Trajectory/shooting/efficiency] features most predictive

#### Model Calibration
- **Calibration Plot**: Predicted probabilities match actual success rates
- **Calibration Improvement**: Brier score improved by X% after calibration

### Scouting Recommendations

#### Top 5 Prospects (2021 Season)

**Rank #1: [Player Name]**
- Age: XX | League: [League] | Team: [Team]
- Stats: XX.X PPG, X.X APG, X.X RPG, XX% 3P%
- Trajectory: +X.X PPG improvement over career
- ML Probability: XX% NBA success likelihood
- Scout Score: XXX.X
- **Why**: [Key strengths - e.g., elite shooter with improving playmaking]

**Rank #2: [Player Name]**
- [Similar format]

**Rank #3: [Player Name]**
- [Similar format]

**Rank #4: [Player Name]**
- [Similar format]

**Rank #5: [Player Name]**
- [Similar format]

#### Prospect Distribution
- With NBA experience: [X] players
- International only: [Y] players
- Average age: XX.X years
- Average scout score: XXX.X

#### Team Fit Analysis
**Top Needs** (based on successful NBA player patterns):
1. **3-Point Shooting**: Weight = 1.XX
2. **Defense**: Weight = 1.XX
3. **Playmaking**: Weight = 1.XX

**Best Fits**: [Players who match team needs]

---

## 🚀 Usage

### Quick Start

```bash
# Navigate to modular directory
cd modular/

# Run full pipeline
python main.py
```

### Running Individual Components

```python
from data.loader import DataLoader
from database.manager import DatabaseManager
from features.statistics import calculate_statistics

# Load data only
loader = DataLoader()
player_df, nba_df, intl_df, issues = loader.load_and_validate()

# Setup database only
db = DatabaseManager()
db.connect().create_schema().create_indexes()
db.load_data(player_df, nba_df, intl_df, issues)

# Calculate statistics only
nba_df, intl_df = calculate_statistics(nba_df, intl_df)
```

### Custom Analysis

```python
from main import ScoutingPipeline

# Initialize pipeline
pipeline = ScoutingPipeline()

# Run specific steps
pipeline.run_data_pipeline()
pipeline.run_statistics_pipeline()
pipeline.run_analysis_pipeline()

# Access results
print(pipeline.traj_df.head())
print(f"Quality issues: {len(pipeline.quality_issues)}")
```

---

## 📝 Configuration

All parameters are in `config.py`:

**To adjust analysis parameters**:
```python
Config.MIN_GAMES_THRESHOLD = 15  # Minimum games for eligibility
Config.MIN_MPG_THRESHOLD = 25    # Minimum minutes per game
Config.MAX_AGE_THRESHOLD = 28    # Maximum age for prospects
```

**To adjust ML parameters**:
```python
Config.ML_N_ESTIMATORS = 150     # More trees (slower, might be better)
Config.ML_MAX_DEPTH = 5          # Deeper trees (risk overfitting)
Config.ML_LEARNING_RATE = 0.05   # Slower learning (might be better)
```

**To use different column groups**:
```python
# For EDA - only correlation columns
eda_cols = Config.SCHEMAS.EDA_NUMERICAL_FOR_CORRELATION

# For ML - all features
ml_features = Config.SCHEMAS.ML_FEATURES_ALL

# For preprocessing - numerical columns
num_cols = Config.SCHEMAS.get_numerical_cols('stats')
```


---

## 📊 Output Files

Generated in root directory:

- `final_scouting_report.csv`: Top 30 prospects with all metrics
- `feature_importance.csv`: ML feature rankings
- `ml_metrics.json`: Detailed model performance metrics
- `data_quality_report.txt`: All data quality issues found
- `eda_visualizations_.png`: 9-panel EDA plot
- `ml_diagnostic_plots.png`: Model calibration, ROC, PR curves
- `kings_scouting.db`: SQLite database with all data

---






In [1]:
%%writefile modular/config.py
"""
 configuration for basketball scouting analysis.
Includes column schemas for automated preprocessing and EDA.
"""

import warnings
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

# Suppress warnings
warnings.filterwarnings('ignore')

# Pandas display options
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 1000)

# Visualization settings
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (18, 12)


class ColumnSchemas:
    """Column schemas for different datasets and purposes."""
    
    # ==================== PLAYER DEMOGRAPHICS ====================
    PLAYER_ID_COLS = [
        'player_id',
        'first_name',
        'last_name'
    ]
    
    PLAYER_DATE_COLS = [
        'birth_date'
    ]
    
    PLAYER_NUMERICAL_DISCRETE = [
        'birth_year',
        'age_2021'
    ]
    
    # ==================== GAME STATISTICS (RAW) ====================
    # Identity columns
    STATS_ID_COLS = [
        'player_id',
        'season',
        'team',
        'league'  # International only
    ]
    
    # Discrete numerical (counts)
    STATS_NUMERICAL_DISCRETE = [
        'games',
        'starts',
        'two_points_made',
        'two_points_attempted',
        'three_points_made',
        'three_points_attempted',
        'free_throws_made',
        'free_throws_attempted',
        'offensive_rebounds',
        'defensive_rebounds',
        'assists',
        'steals',
        'blocked_shots',
        'turnovers',
        'personal_fouls',
        'blocked_shot_attempts',
        'screen_assists',
        'deflections',
        'loose_balls_recovered',
        'personal_fouls_drawn',
        'offensive_fouls',
        'charges_drawn',
        'technical_fouls',
        'flagrant_fouls',
        'ejections',
        'points_off_turnovers',
        'points_in_paint',
        'second_chance_points',
        'fast_break_points'
    ]
    
    # Continuous numerical (measurements)
    STATS_NUMERICAL_CONTINUOUS = [
        'minutes',
        'points',
        'plus_minus',
        'possessions',
        'estimated_possessions',
        'calculated_possessions',
        'plays_used',
        'team_possessions'
    ]
    
    # Percentage/rate columns (0-1 scale)
    STATS_PERCENTAGES = [
        'usage_percentage',
        'true_shooting_percentage',
        'three_point_attempt_rate',
        'free_throw_rate',
        'offensive_rebounding_percentage',
        'defensive_rebounding_percentage',
        'total_rebounding_percentage',
        'assist_percentage',
        'steal_percentage',
        'block_percentage',
        'turnover_percentage'
    ]
    
    # Advanced metrics
    STATS_ADVANCED = [
        'internal_box_plus_minus'
    ]
    
    # Categorical columns
    STATS_CATEGORICAL_NOMINAL = [
        'season_type',  # e.g., "Full Season", "Playoffs"
        'league',       # e.g., "NBA", "EuroLeague", "ACB"
        'team'
    ]
    
    # ==================== CALCULATED STATISTICS ====================
    # Per-game statistics
    CALCULATED_PER_GAME = [
        'ppg',   # points per game
        'apg',   # assists per game
        'rpg',   # rebounds per game
        'spg',   # steals per game
        'bpg',   # blocks per game
        'mpg',   # minutes per game
        'topg',  # turnovers per game
        'fpg'    # fouls per game
    ]
    
    # Per-36 minute statistics
    CALCULATED_PER_36 = [
        'pts_per_36',
        'ast_per_36',
        'reb_per_36'
    ]
    
    # Shooting percentages (calculated)
    CALCULATED_SHOOTING = [
        'fg_pct',        # field goal percentage
        'three_pt_pct',  # three-point percentage
        'ft_pct',        # free throw percentage
        'efg_pct'        # effective field goal percentage
    ]
    
    # Efficiency metrics
    CALCULATED_EFFICIENCY = [
        'efficiency',
        'plus_minus_per_game'
    ]
    
    # All calculated statistics
    CALCULATED_ALL = (
        CALCULATED_PER_GAME + 
        CALCULATED_PER_36 + 
        CALCULATED_SHOOTING + 
        CALCULATED_EFFICIENCY
    )
    
    # ==================== TARGET VARIABLES ====================
    # For ML modeling - NBA success prediction
    ML_TARGET_BINARY = [
        'nba_success'  # Binary: successful NBA career or not
    ]
    
    # For scouting - continuous targets
    SCOUTING_TARGETS = [
        'scout_score',
        'nba_success_prob'
    ]
    
    # ==================== FEATURE GROUPS FOR ML ====================
    # Basic features (readily available)
    ML_FEATURES_BASIC = [
        'ppg', 'apg', 'rpg', 'spg', 'bpg', 'mpg',
        'three_pt_pct', 'ft_pct', 'efficiency'
    ]
    
    # Advanced features (need calculation)
    ML_FEATURES_ADVANCED = [
        'true_shooting_percentage',
        'usage_percentage',
        'efg_pct',
        'ast_to_tov',
        'pts_per_min'
    ]
    
    # Trajectory features (derived from career data)
    ML_FEATURES_TRAJECTORY = [
        'ppg_trend',
        'seasons_played',
        'ppg_change',
        'efficiency_change'
    ]
    
    # All ML features
    ML_FEATURES_ALL = (
        ML_FEATURES_BASIC + 
        ML_FEATURES_ADVANCED + 
        ML_FEATURES_TRAJECTORY + 
        ['games']  # Sample size indicator
    )
    
    # ==================== EDA GROUPINGS ====================
    # For correlation analysis with target
    EDA_NUMERICAL_FOR_CORRELATION = (
        CALCULATED_PER_GAME + 
        CALCULATED_SHOOTING + 
        ['efficiency', 'usage_percentage', 'true_shooting_percentage']
    )
    
    # For distribution analysis
    EDA_DISTRIBUTIONS = CALCULATED_PER_GAME + ['efficiency']
    
    # For comparison across leagues
    EDA_LEAGUE_COMPARISON = [
        'ppg', 'apg', 'rpg', 'efficiency', 
        'three_pt_pct', 'fg_pct', 'mpg'
    ]
    
    @classmethod
    def get_numerical_cols(cls, dataset='stats'):
        """Get all numerical columns for a dataset."""
        if dataset == 'stats':
            return (
                cls.STATS_NUMERICAL_DISCRETE + 
                cls.STATS_NUMERICAL_CONTINUOUS + 
                cls.STATS_PERCENTAGES + 
                cls.STATS_ADVANCED
            )
        elif dataset == 'player':
            return cls.PLAYER_NUMERICAL_DISCRETE
        elif dataset == 'calculated':
            return cls.CALCULATED_ALL
        else:
            return []
    
    @classmethod
    def get_categorical_cols(cls, dataset='stats'):
        """Get all categorical columns for a dataset."""
        if dataset == 'stats':
            return cls.STATS_CATEGORICAL_NOMINAL
        else:
            return []
    
    @classmethod
    def get_id_cols(cls, dataset='stats'):
        """Get ID columns for a dataset."""
        if dataset == 'stats':
            return cls.STATS_ID_COLS
        elif dataset == 'player':
            return cls.PLAYER_ID_COLS
        else:
            return []


class Config:
    """Central configuration for the scouting analysis pipeline."""
    
    # ==================== FILE PATHS ====================
    DATA_DIR = 'api_data_files'
    DB_PATH = 'kings_scouting.db'
    
    # Data files
    PLAYER_FILE = 'player.json'
    NBA_FILE = 'nba_box_player_season.json'
    INTL_FILE = 'international_box_player_season.json'
    
    # Output files
    OUTPUT_DIR = 'outputs'
    SCOUTING_REPORT_CSV = 'final_scouting_report.csv'
    FEATURE_IMPORTANCE_CSV = 'feature_importance.csv'
    ML_METRICS_JSON = 'ml_metrics.json'
    DATA_QUALITY_REPORT = 'data_quality_report.txt'
    EDA_VISUALIZATION = 'eda_visualizations_.png'
    ML_DIAGNOSTIC_PLOTS = 'ml_diagnostic_plots.png'
    
    # ==================== ANALYSIS PARAMETERS ====================
    CURRENT_SEASON = 2021
    MIN_GAMES_THRESHOLD = 10
    MIN_MPG_THRESHOLD = 20
    MAX_AGE_THRESHOLD = 30
    NBA_MIN_GAMES = 10
    
    # NBA success criteria
    NBA_SUCCESS_MIN_MPG = 15
    NBA_SUCCESS_MIN_PPG = 8
    NBA_SUCCESS_MIN_PPG_ALT = 5
    NBA_SUCCESS_MIN_TS = 0.58
    
    # ==================== MODEL PARAMETERS ====================
    ML_TEST_SIZE = 0.25
    ML_RANDOM_STATE = 42
    ML_N_ESTIMATORS = 100
    ML_MAX_DEPTH = 4
    ML_LEARNING_RATE = 0.1
    ML_CV_FOLDS = 5
    ML_N_BOOTSTRAP = 100
    ML_PERM_IMPORTANCE_REPEATS = 10
    
    # ==================== SCOUTING PARAMETERS ====================
    TOP_N_PROSPECTS = 30
    TOP_N_REPORT = 20
    NBA_PROB_WEIGHT = 1.10
    
    # ==================== VALIDATION THRESHOLDS ====================
    MAX_GAMES_PER_SEASON = 100
    MAX_MINUTES_PER_SEASON = 4500
    MIN_AGE = 15
    MAX_AGE_ABSOLUTE = 60
    
    # ==================== DATABASE PRAGMAS ====================
    DB_PRAGMAS = {
        'journal_mode': 'WAL',
        'synchronous': 'NORMAL',
        'foreign_keys': 'ON'
    }
    
    # ==================== REQUIRED COLUMNS ====================
    REQUIRED_PLAYER_COLS = ['first_name', 'last_name', 'birth_date']
    REQUIRED_NBA_COLS = ['season', 'games', 'minutes', 'points', 'assists', 
                         'offensive_rebounds', 'defensive_rebounds']
    REQUIRED_INTL_COLS = ['season', 'games', 'minutes', 'points', 'assists',
                          'offensive_rebounds', 'defensive_rebounds']
    
    # ==================== VALIDATION FIELD GROUPS ====================
    NUMERIC_POSITIVE_FIELDS = ['games', 'minutes', 'points', 'assists', 
                               'steals', 'blocked_shots']
    PERCENTAGE_FIELDS = ['true_shooting_percentage', 'usage_percentage']
    
    SHOT_PAIRS = [
        ('two_points_made', 'two_points_attempted'),
        ('three_points_made', 'three_points_attempted'),
        ('free_throws_made', 'free_throws_attempted')
    ]
    
    # ==================== TEAM WEIGHTS ====================
    DEFAULT_TEAM_WEIGHTS = {
        'shooting_3pt': 1.15,
        'defense': 1.10,
        'playmaking': 1.10,
        'rebounding': 1.05,
        'youth': 1.05
    }
    
    # ==================== PERFORMANCE SCORE WEIGHTS ====================
    PERFORMANCE_WEIGHTS = {
        'ppg': 0.30,
        'efficiency': 0.25,
        'true_shooting_percentage': 0.20,
        'apg': 0.15,
        'rpg': 0.10
    }
    
    # ==================== AGE BONUSES ====================
    AGE_BONUSES = [
        (24, 1.3),
        (26, 1.2),
        (28, 1.1),
        (float('inf'), 1.0)
    ]
    
    # ==================== IMPROVEMENT BONUSES ====================
    IMPROVEMENT_BONUSES = [
        (5, 1.2),
        (3, 1.15),
        (1, 1.1),
        (float('-inf'), 1.0)
    ]
    
    # ==================== COLUMN SCHEMAS ====================
    # Import column schemas
    SCHEMAS = ColumnSchemas


if __name__ == "__main__":
    """Test configuration and schemas."""
    print("=" * 100)
    print("CONFIGURATION & SCHEMA TEST")
    print("=" * 100)
    
    # Test basic config access
    print(f"\n1. Basic Configuration:")
    print(f"   Data Directory: {Config.DATA_DIR}")
    print(f"   Current Season: {Config.CURRENT_SEASON}")
    print(f"   ML Random State: {Config.ML_RANDOM_STATE}")
    
    # Test column schemas
    print(f"\n2. Column Schemas:")
    print(f"   Player ID columns: {len(Config.SCHEMAS.PLAYER_ID_COLS)}")
    print(f"   Stats numerical discrete: {len(Config.SCHEMAS.STATS_NUMERICAL_DISCRETE)}")
    print(f"   Stats numerical continuous: {len(Config.SCHEMAS.STATS_NUMERICAL_CONTINUOUS)}")
    print(f"   Calculated statistics: {len(Config.SCHEMAS.CALCULATED_ALL)}")
    print(f"   ML features: {len(Config.SCHEMAS.ML_FEATURES_ALL)}")
    
    # Test schema methods
    print(f"\n3. Schema Methods:")
    num_cols = Config.SCHEMAS.get_numerical_cols('stats')
    print(f"   All numerical stats columns: {len(num_cols)}")
    cat_cols = Config.SCHEMAS.get_categorical_cols('stats')
    print(f"   All categorical stats columns: {len(cat_cols)}")
    
    # Display some column groups
    print(f"\n4. Key Column Groups:")
    print(f"   Per-game stats: {Config.SCHEMAS.CALCULATED_PER_GAME}")
    print(f"   Shooting percentages: {Config.SCHEMAS.CALCULATED_SHOOTING}")
    print(f"   ML basic features: {Config.SCHEMAS.ML_FEATURES_BASIC[:5]}...")
    print(f"   EDA correlation columns: {len(Config.SCHEMAS.EDA_NUMERICAL_FOR_CORRELATION)}")
    
    # Verify no duplicates in column lists
    print(f"\n5. Validation:")
    all_calc = Config.SCHEMAS.CALCULATED_ALL
    if len(all_calc) == len(set(all_calc)):
        print(f"   ✓ No duplicates in calculated columns")
    else:
        print(f"   ✗ Duplicates found in calculated columns")
    
    ml_features = Config.SCHEMAS.ML_FEATURES_ALL
    if len(ml_features) == len(set(ml_features)):
        print(f"   ✓ No duplicates in ML features")
    else:
        print(f"   ✗ Duplicates found in ML features")
    
    print("\n" + "=" * 100)
    print("CONFIGURATION TEST: PASSED")
    print("=" * 100)

Overwriting modular/config.py


In [2]:
%%writefile modular/utils.py
"""
Utility functions used across the scouting analysis pipeline.
All functions are self-contained and thoroughly tested.
"""

import pandas as pd
import numpy as np
from typing import Union, Optional


from modular.config import Config


def safe_div(numerator: Union[pd.Series, float], 
             denominator: Union[pd.Series, float]) -> Union[pd.Series, float]:
    """
    Safe division that returns NaN for zero or NaN denominator.

    Args:
        numerator: Numerator value(s)
        denominator: Denominator value(s)

    Returns:
        Division result with NaN for invalid divisions
    """
    return np.where(
        (pd.notna(denominator)) & (denominator != 0), 
        numerator / denominator, 
        np.nan
    )


def normalize_metric(series: pd.Series) -> pd.Series:
    """
    Normalize a metric to 0-100 scale using min-max scaling.

    Args:
        series: Series of values to normalize

    Returns:
        Normalized series (0-100 scale) or NaN if cannot normalize
    """
    series = series.astype(float)
    mn, mx = series.min(), series.max()

    if pd.isna(mn) or pd.isna(mx) or mx <= mn:
        return pd.Series(np.nan, index=series.index)

    return (series - mn) / (mx - mn) * 100.0


def calculate_efg_percentage(two_pm: pd.Series, three_pm: pd.Series,
                             two_pa: pd.Series, three_pa: pd.Series) -> pd.Series:
    """
    Calculate effective field goal percentage.

    eFG% = (2PM + 1.5 * 3PM) / (2PA + 3PA)

    Args:
        two_pm: Two-point makes
        three_pm: Three-point makes
        two_pa: Two-point attempts
        three_pa: Three-point attempts

    Returns:
        Series of effective FG percentages
    """
    total_fga = two_pa.fillna(0) + three_pa.fillna(0)
    weighted_makes = two_pm.fillna(0) + 1.5 * three_pm.fillna(0)

    return safe_div(weighted_makes, total_fga)


def calculate_age_bonus(age: float) -> float:
    """
    Calculate age bonus multiplier.

    Younger players get higher multipliers due to greater upside potential.

    Args:
        age: Player age

    Returns:
        Age bonus multiplier
        
    Examples:
        >>> calculate_age_bonus(22)
        1.3
        >>> calculate_age_bonus(27)
        1.1
        >>> calculate_age_bonus(30)
        1.0
    """
    if age < 24:
        return 1.3
    elif age < 26:
        return 1.2
    elif age < 28:
        return 1.1
    else:
        return 1.0


def calculate_improvement_bonus(ppg_change: float) -> float:
    """
    Calculate improvement bonus multiplier based on PPG trend.

    Args:
        ppg_change: Change in PPG from first to last season

    Returns:
        Improvement bonus multiplier
        
    Examples:
        >>> calculate_improvement_bonus(6.0)
        1.2
        >>> calculate_improvement_bonus(4.0)
        1.15
        >>> calculate_improvement_bonus(0.5)
        1.0
    """
    if ppg_change > 5:
        return 1.2
    elif ppg_change > 3:
        return 1.15
    elif ppg_change > 1:
        return 1.1
    else:
        return 1.0


def create_player_id(first_name: pd.Series, last_name: pd.Series,
                     birth_year: Optional[pd.Series] = None) -> pd.Series:
    """
    Create player ID from name components.

    Args:
        first_name: Series of first names
        last_name: Series of last names
        birth_year: Optional series of birth years (for collision resolution)

    Returns:
        Series of player IDs
    """
    base_id = first_name.str.lower() + '_' + last_name.str.lower()

    if birth_year is not None:
        return base_id + '_' + birth_year.astype(str)

    return base_id


def check_numeric_bounds(series: pd.Series, min_val: float, max_val: float,
                        name: str = 'value') -> pd.Series:
    """
    Check if numeric values are within expected bounds.

    Args:
        series: Series to check
        min_val: Minimum valid value
        max_val: Maximum valid value
        name: Name for reporting

    Returns:
        Series with out-of-bounds values set to NaN
    """
    out_of_bounds = (series < min_val) | (series > max_val)

    if out_of_bounds.any():
        print(f"Warning: {out_of_bounds.sum()} out-of-bounds values in {name}")
        series = series.copy()
        series[out_of_bounds] = np.nan

    return series


def format_stat(value, format_type: str = 'float') -> str:
    """
    Format statistical value for display.

    Args:
        value: Value to format
        format_type: Type of formatting ('int', 'float1', 'float2', 'pct', 'str')

    Returns:
        Formatted string
    """
    if pd.isna(value) or value is None:
        return "—"

    if format_type == 'int':
        return f"{int(value)}"
    elif format_type == 'float1':
        return f"{value:.1f}"
    elif format_type == 'float2':
        return f"{value:.2f}"
    elif format_type == 'pct':
        return f"{value:.1%}"
    elif format_type == 'str':
        return str(value)
    else:
        return str(value)


def format_probability_range(prob: float, ci_low: Optional[float] = None,
                            ci_high: Optional[float] = None) -> str:
    """
    Format probability with optional confidence interval.

    Args:
        prob: Probability value
        ci_low: Optional lower CI bound
        ci_high: Optional upper CI bound

    Returns:
        Formatted probability string
    """
    if pd.isna(prob):
        return "—"

    if ci_low is not None and ci_high is not None and not (pd.isna(ci_low) or pd.isna(ci_high)):
        return f"{prob:.1%} ({ci_low:.1%}–{ci_high:.1%})"

    if prob >= 0.95:
        return f">{prob*0.9:.0%}"
    elif prob <= 0.05:
        return f"<{prob*2:.0%}"
    else:
        return f"{prob:.1%}"


def merge_trajectory_features(df: pd.DataFrame, traj_df: pd.DataFrame,
                              on: str = 'player_id') -> pd.DataFrame:
    """
    Merge trajectory features into a DataFrame.

    Args:
        df: DataFrame to merge into
        traj_df: Trajectory DataFrame
        on: Column to merge on

    Returns:
        Merged DataFrame
    """
    if traj_df is None or traj_df.empty:
        return df

    traj_cols = ['ppg_change', 'efficiency_change', 'seasons_played']
    available_cols = [col for col in traj_cols if col in traj_df.columns]

    if not available_cols:
        return df

    return df.merge(
        traj_df[available_cols].reset_index(),
        on=on,
        how='left'
    )


def calculate_career_stats(df: pd.DataFrame, group_by: str = 'player_id',
                          stats: list = ['ppg', 'apg', 'rpg']) -> pd.DataFrame:
    """
    Calculate career aggregate statistics.

    Args:
        df: DataFrame with player statistics
        group_by: Column to group by
        stats: List of statistics to aggregate

    Returns:
        DataFrame with career statistics
    """
    agg_dict = {}

    for stat in stats:
        if stat in df.columns:
            agg_dict[stat] = ['mean', 'max', 'min', 'std']

    if not agg_dict:
        return pd.DataFrame()

    agg_dict['season'] = ['count', 'min', 'max']

    career = df.groupby(group_by).agg(agg_dict)
    career.columns = ['_'.join(col).strip() for col in career.columns.values]
    career = career.rename(columns={'season_count': 'seasons_played'})

    return career.reset_index()


def validate_dataframe(df: pd.DataFrame, required_cols: list, 
                      name: str = 'DataFrame') -> tuple:
    """
    Validate DataFrame has required columns and report issues.

    Args:
        df: DataFrame to validate
        required_cols: List of required column names
        name: Name for reporting

    Returns:
        Tuple of (is_valid, issues_list)
    """
    issues = []

    # Check required columns
    missing = [col for col in required_cols if col not in df.columns]
    if missing:
        issues.append(f"{name}: Missing columns {missing}")

    # Check for empty
    if len(df) == 0:
        issues.append(f"{name}: Empty DataFrame")

    # Check for duplicate indices
    if df.index.duplicated().any():
        issues.append(f"{name}: Duplicate indices found")

    return len(issues) == 0, issues


if __name__ == "__main__":
    """Smoke test for utility functions."""
    print("=" * 100)
    print("UTILS MODULE SMOKE TEST")
    print("=" * 100)
    
    try:
        # Test safe_div
        print("\n1. Testing safe_div...")
        assert safe_div(10, 2) == 5.0, "Basic division failed"
        assert np.isnan(safe_div(10, 0)), "Division by zero should return NaN"
        result = safe_div(pd.Series([10, 20, 30]), pd.Series([2, 0, 3]))
        assert result[0] == 5.0, "Series division failed"
        assert np.isnan(result[1]), "Series division by zero failed"
        print("✓ safe_div works correctly")
        
        # Test normalize_metric
        print("\n2. Testing normalize_metric...")
        series = pd.Series([1, 2, 3, 4, 5])
        normalized = normalize_metric(series)
        assert normalized.min() == 0.0, "Min should be 0"
        assert normalized.max() == 100.0, "Max should be 100"
        assert abs(normalized.iloc[2] - 50.0) < 0.01, "Middle should be 50"
        print("✓ normalize_metric works correctly")
        
        # Test calculate_efg_percentage
        print("\n3. Testing calculate_efg_percentage...")
        two_pm = pd.Series([10])
        three_pm = pd.Series([5])
        two_pa = pd.Series([20])
        three_pa = pd.Series([15])
        efg = calculate_efg_percentage(two_pm, three_pm, two_pa, three_pa)
        # (10 + 1.5*5) / (20 + 15) = 17.5 / 35 = 0.5
        assert abs(efg.iloc[0] - 0.5) < 0.01, "eFG% calculation incorrect"
        print("✓ calculate_efg_percentage works correctly")
        
        # Test create_player_id
        print("\n4. Testing create_player_id...")
        pid = create_player_id(pd.Series(['John']), pd.Series(['Doe']))
        assert pid.iloc[0] == 'john_doe', "Simple player ID failed"
        pid_with_year = create_player_id(pd.Series(['John']), pd.Series(['Doe']), pd.Series([1990]))
        assert pid_with_year.iloc[0] == 'john_doe_1990', "Player ID with year failed"
        print("✓ create_player_id works correctly")
        
        # Test age bonus
        print("\n5. Testing calculate_age_bonus...")
        assert calculate_age_bonus(22) == 1.3, "Young age bonus incorrect"
        assert calculate_age_bonus(25) == 1.2, "Mid age bonus incorrect"
        assert calculate_age_bonus(30) == 1.0, "Old age bonus incorrect"
        print("✓ calculate_age_bonus works correctly")
        
        # Test improvement bonus
        print("\n6. Testing calculate_improvement_bonus...")
        assert calculate_improvement_bonus(6.0) == 1.2, "High improvement bonus incorrect"
        assert calculate_improvement_bonus(4.0) == 1.15, "Mid improvement bonus incorrect"
        assert calculate_improvement_bonus(0.5) == 1.0, "Low improvement bonus incorrect"
        print("✓ calculate_improvement_bonus works correctly")
        
        # Test format_stat
        print("\n7. Testing format_stat...")
        assert format_stat(10.5, 'int') == '10', "Int format failed"
        assert format_stat(10.567, 'float1') == '10.6', "Float1 format failed"
        assert format_stat(0.456, 'pct') == '45.6%', "Pct format failed"
        assert format_stat(None, 'float') == '—', "None format failed"
        print("✓ format_stat works correctly")
        
        # Test format_probability_range
        print("\n8. Testing format_probability_range...")
        assert format_probability_range(0.75) == '75.0%', "Basic prob format failed"
        assert '65.0%' in format_probability_range(0.75, 0.65, 0.85), "CI format failed"
        assert '>' in format_probability_range(0.96), "High prob format failed"
        print("✓ format_probability_range works correctly")
        
        print("\n" + "=" * 100)
        print("UTILS MODULE: ALL TESTS PASSED ✓")
        print("=" * 100)
        
    except AssertionError as e:
        print(f"\n✗ TEST FAILED: {e}")
        print("\n" + "=" * 100)
        print("UTILS MODULE: FAILED ✗")
        print("=" * 100)
    except Exception as e:
        print(f"\n✗ ERROR: {e}")
        import traceback
        traceback.print_exc()
        print("\n" + "=" * 100)
        print("UTILS MODULE: FAILED ✗")
        print("=" * 100)


Overwriting modular/utils.py


In [3]:
%%writefile modular/data/loader.py
"""
Data loading and validation module - FIXED VERSION
Addresses: Usage % scale issue, consistent issue counting
"""

import json
import pandas as pd
import numpy as np
from typing import List, Tuple, Dict
from pathlib import Path

from modular.config import Config
from modular.utils import safe_div, create_player_id



class DataLoader:
    """Handles loading and validation of player data from JSON files."""

    def __init__(self, data_dir: str = None):
        self.data_dir = data_dir or Config.DATA_DIR
        self.player_df = None
        self.nba_df = None
        self.intl_df = None
        # FIXED: Track both issue types AND affected row counts
        self.data_quality_issues = []  # List of issue descriptions
        self.issue_stats = {
            'total_issue_types': 0,
            'total_rows_affected': 0,
            'by_dataset': {}
        }

    def _log_issue(self, dataset: str, issue_type: str, description: str, rows_affected: int = 0):
        """
        Centralized issue logging with consistent counting.

        Args:
            dataset: Dataset name (Player/NBA/International)
            issue_type: Type of issue (duplicate_names, out_of_range, etc.)
            description: Full description
            rows_affected: Number of rows affected
        """
        self.data_quality_issues.append(description)
        self.issue_stats['total_issue_types'] += 1
        self.issue_stats['total_rows_affected'] += rows_affected

        if dataset not in self.issue_stats['by_dataset']:
            self.issue_stats['by_dataset'][dataset] = {
                'issue_types': 0,
                'rows_affected': 0
            }
        self.issue_stats['by_dataset'][dataset]['issue_types'] += 1
        self.issue_stats['by_dataset'][dataset]['rows_affected'] += rows_affected

    def load_raw_data(self) -> 'DataLoader':
        """Load raw JSON data files into DataFrames."""
        print("=" * 100)
        print("LOADING DATA")
        print("=" * 100)

        with open(Path(self.data_dir) / Config.PLAYER_FILE, 'r') as f:
            self.player_df = pd.DataFrame(json.load(f))
        with open(Path(self.data_dir) / Config.NBA_FILE, 'r') as f:
            self.nba_df = pd.DataFrame(json.load(f))
        with open(Path(self.data_dir) / Config.INTL_FILE, 'r') as f:
            self.intl_df = pd.DataFrame(json.load(f))

        print(f"Loaded {len(self.player_df):,} player rows")
        print(f"Loaded {len(self.nba_df):,} NBA player-seasons")
        print(f"Loaded {len(self.intl_df):,} International player-seasons")
        return self

    def process_demographics(self) -> 'DataLoader':
        """Process player demographics including birth dates and ages."""
        print("\nProcessing demographics...")
        self.player_df['birth_date'] = pd.to_datetime(self.player_df['birth_date'])
        self.player_df['birth_year'] = self.player_df['birth_date'].dt.year
        # FIXED: Age calculated as of 2021 (not current date)
        self.player_df['age_2021'] = 2021 - self.player_df['birth_year']
        print(f"Age range (as of 2021): {self.player_df['age_2021'].min():.0f}–{self.player_df['age_2021'].max():.0f}")
        return self

    def create_player_ids(self) -> 'DataLoader':
        """Create initial player IDs for all datasets."""
        print("\nCreating player IDs...")
        for df in [self.player_df, self.nba_df, self.intl_df]:
            df['player_id'] = (df['first_name'].str.lower() + '_' + df['last_name'].str.lower())
        return self

    def check_duplicate_names(self, df: pd.DataFrame, name: str) -> pd.DataFrame:
        """Check for duplicate player names and report them."""
        name_counts = df.groupby(['first_name', 'last_name']).size()
        duplicates = name_counts[name_counts > 1]

        if len(duplicates) > 0:
            rows_affected = duplicates.sum()
            issue = f"[{name}] Found {len(duplicates)} duplicate name(s) affecting {rows_affected} rows:"
            for (fname, lname), count in duplicates.items():
                issue += f"\n  - {fname} {lname}: {count} occurrences"

            self._log_issue(name, 'duplicate_names', issue, rows_affected)
            print(issue)

            if 'birth_year' in df.columns:
                print(f"[{name}] Enhancing player_id with birth_year to resolve collisions")
                df['player_id'] = create_player_id(df['first_name'], df['last_name'], df['birth_year'])

        return df

    def detect_collisions(self) -> 'DataLoader':
        """Detect and resolve player name collisions across datasets."""
        print("\n" + "=" * 100)
        print("CHECKING FOR DUPLICATE NAMES")
        print("=" * 100)

        self.player_df = self.check_duplicate_names(self.player_df, 'Player')
        nba_temp = self.nba_df[['first_name', 'last_name']].drop_duplicates()
        self.check_duplicate_names(nba_temp, 'NBA')
        intl_temp = self.intl_df[['first_name', 'last_name']].drop_duplicates()
        self.check_duplicate_names(intl_temp, 'International')

        if any('birth_year' in issue for issue in self.data_quality_issues):
            print("\nUpdating player_id in NBA and International data...")
            player_id_map = self.player_df.set_index(
                self.player_df['first_name'].str.lower() + '_' + self.player_df['last_name'].str.lower()
            )['player_id'].to_dict()

            simple_id = self.nba_df['first_name'].str.lower() + '_' + self.nba_df['last_name'].str.lower()
            self.nba_df['player_id'] = simple_id.map(player_id_map).fillna(simple_id)

            simple_id = self.intl_df['first_name'].str.lower() + '_' + self.intl_df['last_name'].str.lower()
            self.intl_df['player_id'] = simple_id.map(player_id_map).fillna(simple_id)

        return self

    def validate_data(self, df: pd.DataFrame, name: str, required_cols: List[str]) -> pd.DataFrame:
        """ validation with usage% scale fix."""
        print(f"\n[{name}] Validating data...")

        # Check required columns
        missing = [col for col in required_cols if col not in df.columns]
        if missing:
            issue = f"[{name}] Missing required columns (adding as NaN): {missing}"
            self._log_issue(name, 'missing_columns', issue, len(df))
            print(issue)
            for col in missing:
                df[col] = np.nan

        # Validate numeric positive fields
        for col in Config.NUMERIC_POSITIVE_FIELDS:
            if col not in df.columns:
                continue

            invalid_count = 0
            negative_mask = df[col] < 0
            if negative_mask.any():
                invalid_count += negative_mask.sum()
                df.loc[negative_mask, col] = np.nan

            if col == 'games':
                extreme_mask = df[col] > Config.MAX_GAMES_PER_SEASON
                if extreme_mask.any():
                    invalid_count += extreme_mask.sum()

                    # DEBUG ISSUE #5: Show which players have games > 100
                    print(f"\n[DEBUG ISSUE #5] Found {extreme_mask.sum()} records with games > {Config.MAX_GAMES_PER_SEASON}:")
                    extreme_records = df[extreme_mask][['player_id', 'first_name', 'last_name', 'season', 'games', 'minutes', 'points']].head(10)
                    for idx, row in extreme_records.iterrows():
                        print(f"  {row['first_name']} {row['last_name']} ({row['season']}): games={row['games']}, min={row.get('minutes', 'N/A')}, pts={row.get('points', 'N/A')}")
                    if extreme_mask.sum() > 10:
                        print(f"  ... and {extreme_mask.sum() - 10} more")

                    issue = f"[{name}] {extreme_mask.sum()} extreme 'games' values (>{Config.MAX_GAMES_PER_SEASON}) - SET TO NaN"
                    self._log_issue(name, 'extreme_values', issue, extreme_mask.sum())
                    print(f"[DEBUG ISSUE #5] Setting games to NaN for these records (will invalidate per-game stats)")

                    # Also invalidate dependent stats since games is unreliable
                    df.loc[extreme_mask, col] = np.nan
                    # Note: Don't invalidate other columns yet - let user decide

            if col == 'minutes':
                extreme_mask = df[col] > Config.MAX_MINUTES_PER_SEASON
                if extreme_mask.any():
                    invalid_count += extreme_mask.sum()
                    issue = f"[{name}] {extreme_mask.sum()} extreme 'minutes' values (>{Config.MAX_MINUTES_PER_SEASON})"
                    self._log_issue(name, 'extreme_values', issue, extreme_mask.sum())
                    print(issue)
                    df.loc[extreme_mask, col] = np.nan

            if invalid_count > 0 and col in ['games', 'minutes', 'points']:
                issue = f"[{name}] Total invalid '{col}' values: {invalid_count}"
                self._log_issue(name, 'invalid_numeric', issue, invalid_count)
                print(issue)

        # FIXED: Validate percentage fields with scale detection
        for col in Config.PERCENTAGE_FIELDS:
            if col not in df.columns:
                continue

            # Check if values are in 0-100 scale (should be 0-1)
            if col == 'usage_percentage' and (df[col] > 1.5).sum() > len(df) * 0.5:
                # More than 50% of values > 1.5 suggests 0-100 scale
                print(f"[{name}] Detected {col} in 0-100 scale, converting to 0-1 scale")
                df[col] = df[col] / 100.0
                issue = f"[{name}] Converted {col} from 0-100 to 0-1 scale"
                self._log_issue(name, 'scale_conversion', issue, len(df[df[col].notna()]))

            # Now validate 0-1 range
            out_of_range = ((df[col] < 0) | (df[col] > 1)).fillna(False)
            if out_of_range.any():
                issue = f"[{name}] {out_of_range.sum()} out-of-range '{col}' values (should be 0-1)"
                self._log_issue(name, 'out_of_range', issue, out_of_range.sum())
                print(issue)
                df.loc[out_of_range, col] = np.nan

        # Validate shooting logic
        for made_col, att_col in Config.SHOT_PAIRS:
            if made_col not in df.columns or att_col not in df.columns:
                continue

            invalid = ((df[made_col] > df[att_col]) & df[made_col].notna() & df[att_col].notna())
            if invalid.any():
                issue = f"[{name}] {invalid.sum()} rows where {made_col} > {att_col}"
                self._log_issue(name, 'invalid_shooting', issue, invalid.sum())
                print(issue)
                df.loc[invalid, [made_col, att_col]] = np.nan

        # Check for zero-stat records
        if all(col in df.columns for col in ['games', 'minutes', 'points']):
            zero_stats = (
                (df['games'].fillna(0) == 0) & 
                (df['minutes'].fillna(0) == 0) & 
                (df['points'].fillna(0) == 0)
            )
            if zero_stats.any():
                issue = f"[{name}] {zero_stats.sum()} records with zero games/minutes/points"
                self._log_issue(name, 'zero_stats', issue, zero_stats.sum())
                print(issue)

        print(f"[{name}] Validation complete.")
        return df

    def validate_all_data(self) -> 'DataLoader':
        """Validate all datasets."""
        print("\n" + "=" * 100)
        print("VALIDATING DATA")
        print("=" * 100)

        self.player_df = self.validate_data(self.player_df, 'Player', Config.REQUIRED_PLAYER_COLS)
        self.nba_df = self.validate_data(self.nba_df, 'NBA', Config.REQUIRED_NBA_COLS)
        self.intl_df = self.validate_data(self.intl_df, 'International', Config.REQUIRED_INTL_COLS)
        return self

    def print_summary(self) -> 'DataLoader':
        """FIXED: Print consistent data quality summary with detailed breakdown."""
        print("\n" + "=" * 100)
        print("DATA QUALITY SUMMARY")
        print("=" * 100)
        print(f"Total issue types identified: {self.issue_stats['total_issue_types']}")
        print(f"Total rows affected: {self.issue_stats['total_rows_affected']}")

        # DEBUG ISSUE #2: Breakdown by issue type
        print(f"\n[DEBUG ISSUE #2] Breakdown by issue category:")
        issue_type_counts = {}
        for issue in self.data_quality_issues:
            # Categorize issues
            if 'scale' in issue.lower() or 'converted' in issue.lower():
                category = 'scale_conversion'
            elif 'out-of-range' in issue.lower() or 'out_of_range' in issue:
                category = 'out_of_range'
            elif 'extreme' in issue.lower():
                category = 'extreme_values'
            elif 'duplicate' in issue.lower():
                category = 'duplicate_names'
            elif 'orphan' in issue.lower() or 'filtered' in issue.lower():
                category = 'orphaned_records'
            elif 'invalid' in issue.lower():
                category = 'invalid_values'
            elif 'missing' in issue.lower():
                category = 'missing_columns'
            elif 'zero' in issue.lower():
                category = 'zero_stats'
            else:
                category = 'other'

            issue_type_counts[category] = issue_type_counts.get(category, 0) + 1

        for category, count in sorted(issue_type_counts.items(), key=lambda x: x[1], reverse=True):
            print(f"  {category}: {count} issue(s)")

        print("\nBy dataset:")
        for dataset, stats in self.issue_stats['by_dataset'].items():
            # Calculate percentage of dataset
            if dataset == 'NBA':
                total_rows = len(self.nba_df)
            elif dataset == 'International':
                total_rows = len(self.intl_df)
            elif dataset == 'Player':
                total_rows = len(self.player_df)
            else:
                total_rows = 0

            pct = (stats['rows_affected'] / total_rows * 100) if total_rows > 0 else 0
            print(f"  {dataset}: {stats['issue_types']} issue type(s), {stats['rows_affected']} row(s) affected ({pct:.1f}% of dataset)")

        # DEBUG ISSUE #2: Show sample issues
        print(f"\n[DEBUG ISSUE #2] Sample issues (first 5):")
        for i, issue in enumerate(self.data_quality_issues[:5], 1):
            # Truncate long issues
            issue_short = issue[:120] + '...' if len(issue) > 120 else issue
            print(f"  {i}. {issue_short}")

        return self

    def load_and_validate(self) -> Tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame, List[str]]:
        """Complete data loading and validation pipeline."""
        self.load_raw_data()
        self.process_demographics()
        self.create_player_ids()
        self.detect_collisions()
        self.validate_all_data()
        self.print_summary()
        return self.player_df, self.nba_df, self.intl_df, self.data_quality_issues


if __name__ == "__main__":
    """Smoke test with synthetic data."""
    print("="*100)
    print("DATA LOADER SMOKE TEST (FIXED VERSION)")
    print("="*100)

    import tempfile, shutil
    temp_dir = tempfile.mkdtemp()

    try:
        # Create test data with usage_percentage in wrong scale
        test_players = [{"first_name": "Test", "last_name": "Player1", "birth_date": "1995-01-15"}]
        test_nba = [{
            "first_name": "Test", "last_name": "Player1", "season": 2021,
            "games": 50, "minutes": 1200, "points": 600, "assists": 200,
            "offensive_rebounds": 50, "defensive_rebounds": 150,
            "usage_percentage": 25.5  # Wrong scale (0-100 instead of 0-1)
        }]
        test_intl = []

        for fname, data in [
            ('player.json', test_players),
            ('nba_box_player_season.json', test_nba),
            ('international_box_player_season.json', test_intl)
        ]:
            with open(Path(temp_dir) / fname, 'w') as f:
                json.dump(data, f)

        loader = DataLoader(data_dir=temp_dir)
        player_df, nba_df, intl_df, issues = loader.load_and_validate()

        # Check usage_percentage was converted
        assert nba_df['usage_percentage'].iloc[0] == 0.255, "Usage % not converted correctly"
        print("\n✓ Usage percentage scale conversion works")

        # Check issue counting
        assert loader.issue_stats['total_issue_types'] > 0, "Should have logged issues"
        print(f"✓ Issue tracking works: {loader.issue_stats['total_issue_types']} types, {loader.issue_stats['total_rows_affected']} rows")

        shutil.rmtree(temp_dir)
        print("\n✓ DATA LOADER (FIXED): PASSED")

    except Exception as e:
        print(f"\n✗ ERROR: {e}")
        import traceback
        traceback.print_exc()


Overwriting modular/data/loader.py


In [4]:
%%writefile modular/database/manager.py
"""
Database operations module for basketball scouting analysis.
Handles SQLite database setup, schema creation, and ETL operations.
"""

import sqlite3
import pandas as pd
from typing import List, Optional
from pathlib import Path

from modular.config import Config



class DatabaseManager:
    """Manages SQLite database operations for scouting data."""

    def __init__(self, db_path: str = None):
        """
        Initialize DatabaseManager.

        Args:
            db_path: Path to SQLite database file (default from Config)
        """
        self.db_path = db_path or Config.DB_PATH
        self.conn = None

    def connect(self) -> 'DatabaseManager':
        """Create database connection and apply pragmas."""
        self.conn = sqlite3.connect(self.db_path)

        # Apply pragmas for performance and reliability
        for pragma, value in Config.DB_PRAGMAS.items():
            self.conn.execute(f"PRAGMA {pragma}={value};")

        print(f"SQLite database connected: {self.db_path}")
        return self

    def create_schema(self) -> 'DatabaseManager':
        """Create database schema with constraints."""
        assert self.conn is not None, "Database not connected"

        print("Creating database schema...")

        # Disable foreign keys temporarily for schema creation
        self.conn.execute("PRAGMA foreign_keys=OFF;")

        # Create tables
        self.conn.executescript("""
            -- Players table
            DROP TABLE IF EXISTS players;
            CREATE TABLE players (
                player_id TEXT PRIMARY KEY,
                first_name TEXT NOT NULL,
                last_name TEXT NOT NULL,
                birth_date TEXT,
                birth_year INTEGER,
                age_2021 INTEGER CHECK(age_2021 >= 15 AND age_2021 <= 60)
            );

            -- NBA stats table
            DROP TABLE IF EXISTS nba_stats;
            CREATE TABLE nba_stats (
                player_id TEXT NOT NULL,
                season INTEGER NOT NULL,
                team TEXT,
                games INTEGER CHECK(games > 0),
                minutes REAL CHECK(minutes >= 0),
                points REAL CHECK(points >= 0),
                assists REAL CHECK(assists >= 0),
                offensive_rebounds REAL,
                defensive_rebounds REAL,
                steals REAL,
                blocked_shots REAL,
                turnovers REAL,
                personal_fouls REAL,
                two_points_made REAL,
                two_points_attempted REAL,
                three_points_made REAL,
                three_points_attempted REAL,
                free_throws_made REAL,
                free_throws_attempted REAL,
                true_shooting_percentage REAL,
                usage_percentage REAL,
                plus_minus REAL,
                FOREIGN KEY (player_id) REFERENCES players(player_id),
                PRIMARY KEY (player_id, season, team)
            );

            -- International stats table
            DROP TABLE IF EXISTS intl_stats;
            CREATE TABLE intl_stats (
                player_id TEXT NOT NULL,
                season INTEGER NOT NULL,
                league TEXT NOT NULL,
                team TEXT,
                games INTEGER CHECK(games > 0),
                starts INTEGER,
                minutes REAL CHECK(minutes >= 0),
                points REAL CHECK(points >= 0),
                assists REAL CHECK(assists >= 0),
                offensive_rebounds REAL,
                defensive_rebounds REAL,
                steals REAL,
                blocked_shots REAL,
                turnovers REAL,
                personal_fouls REAL,
                two_points_made REAL,
                two_points_attempted REAL,
                three_points_made REAL,
                three_points_attempted REAL,
                free_throws_made REAL,
                free_throws_attempted REAL,
                true_shooting_percentage REAL,
                usage_percentage REAL,
                FOREIGN KEY (player_id) REFERENCES players(player_id),
                PRIMARY KEY (player_id, season, league, team)
            );

            -- Data quality log table
            DROP TABLE IF EXISTS data_quality_log;
            CREATE TABLE data_quality_log (
                issue_id INTEGER PRIMARY KEY AUTOINCREMENT,
                issue_description TEXT NOT NULL,
                timestamp TEXT DEFAULT CURRENT_TIMESTAMP
            );
        """)

        # Re-enable foreign keys
        self.conn.execute("PRAGMA foreign_keys=ON;")
        self.conn.commit()

        print("Schema created: players, nba_stats, intl_stats, data_quality_log")
        return self

    def create_indexes(self) -> 'DatabaseManager':
        """Create indexes for improved query performance."""
        assert self.conn is not None, "Database not connected"

        print("Creating indexes...")

        self.conn.execute(
            "CREATE INDEX IF NOT EXISTS idx_nba_season ON nba_stats(season);"
        )
        self.conn.execute(
            "CREATE INDEX IF NOT EXISTS idx_intl_season ON intl_stats(season);"
        )
        self.conn.execute(
            "CREATE INDEX IF NOT EXISTS idx_intl_league ON intl_stats(league);"
        )
        self.conn.commit()

        print("Indexes created")
        return self

    def filter_orphaned_stats(self, player_df: pd.DataFrame, 
                             nba_df: pd.DataFrame,
                             intl_df: pd.DataFrame,
                             quality_issues: List[str]) -> tuple:
        """
        Remove stats records without corresponding player demographics.

        Args:
            player_df: Player demographics DataFrame
            nba_df: NBA stats DataFrame
            intl_df: International stats DataFrame
            quality_issues: List to append quality issues to

        Returns:
            Tuple of (filtered_nba_df, filtered_intl_df, updated_quality_issues)
        """
        player_ids = set(player_df['player_id'].unique())

        # Check NBA orphans
        nba_orphans = set(nba_df['player_id'].unique()) - player_ids
        if len(nba_orphans) > 0:
            before = len(nba_df)
            orphan_names = nba_df[nba_df['player_id'].isin(nba_orphans)][
                ['first_name', 'last_name']
            ].drop_duplicates()

            issue = (f"Filtered {before - len(nba_df[nba_df['player_id'].isin(player_ids)])} "
                    f"NBA rows without demographics: "
                    f"{list(orphan_names.itertuples(index=False, name=None))}")
            quality_issues.append(issue)
            print(issue)

            nba_df = nba_df[nba_df['player_id'].isin(player_ids)].copy()

        # Check International orphans
        intl_orphans = set(intl_df['player_id'].unique()) - player_ids
        if len(intl_orphans) > 0:
            before = len(intl_df)
            orphan_names = intl_df[intl_df['player_id'].isin(intl_orphans)][
                ['first_name', 'last_name']
            ].drop_duplicates()

            issue = (f"Filtered {before - len(intl_df[intl_df['player_id'].isin(player_ids)])} "
                    f"International rows without demographics: "
                    f"{list(orphan_names.itertuples(index=False, name=None))}")
            quality_issues.append(issue)
            print(issue)

            intl_df = intl_df[intl_df['player_id'].isin(player_ids)].copy()

        return nba_df, intl_df, quality_issues

    def load_data(self, player_df: pd.DataFrame,
                  nba_df: pd.DataFrame,
                  intl_df: pd.DataFrame,
                  quality_issues: List[str]) -> tuple:
        """
        Load data into database tables.

        DEBUG ENHANCEMENT: Now returns filtered DataFrames for pipeline consistency.

        Args:
            player_df: Player demographics DataFrame
            nba_df: NBA stats DataFrame
            intl_df: International stats DataFrame
            quality_issues: List of data quality issues to log

        Returns:
            Tuple of (filtered_nba_df, filtered_intl_df) for downstream use
        """
        assert self.conn is not None, "Database not connected"

        print("\nLoading data into database...")

        # DEBUG: Show input counts
        print(f"[DEBUG ISSUE #1] Input counts to database:")
        print(f"  player_df: {len(player_df)} rows")
        print(f"  nba_df: {len(nba_df)} rows")
        print(f"  intl_df: {len(intl_df)} rows")

        # Filter orphaned records
        nba_df, intl_df, quality_issues = self.filter_orphaned_stats(
            player_df, nba_df, intl_df, quality_issues
        )

        # DEBUG: Show filtered counts
        print(f"[DEBUG ISSUE #1] After orphan filtering:")
        print(f"  nba_df: {len(nba_df)} rows (filtered)")
        print(f"  intl_df: {len(intl_df)} rows (filtered)")

        # Insert players
        player_cols = ['player_id', 'first_name', 'last_name', 
                      'birth_date', 'birth_year', 'age_2021']
        player_df[player_cols].drop_duplicates(subset='player_id').to_sql(
            'players', self.conn, if_exists='append', index=False
        )
        print(f"Loaded {len(player_df)} players")

        # Insert NBA stats
        nba_cols = ['player_id', 'season', 'team', 'games', 'minutes', 'points', 
                   'assists', 'offensive_rebounds', 'defensive_rebounds', 'steals', 
                   'blocked_shots', 'turnovers', 'personal_fouls', 'two_points_made', 
                   'two_points_attempted', 'three_points_made', 'three_points_attempted',
                   'free_throws_made', 'free_throws_attempted', 'true_shooting_percentage', 
                   'usage_percentage', 'plus_minus']
        nba_df[[c for c in nba_cols if c in nba_df.columns]].to_sql(
            'nba_stats', self.conn, if_exists='append', index=False
        )
        print(f"Loaded {len(nba_df)} NBA records")

        # Insert International stats
        intl_cols = ['player_id', 'season', 'league', 'team', 'games', 'starts', 
                    'minutes', 'points', 'assists', 'offensive_rebounds', 
                    'defensive_rebounds', 'steals', 'blocked_shots', 'turnovers', 
                    'personal_fouls', 'two_points_made', 'two_points_attempted',
                    'three_points_made', 'three_points_attempted', 'free_throws_made',
                    'free_throws_attempted', 'true_shooting_percentage', 'usage_percentage']
        intl_df[[c for c in intl_cols if c in intl_df.columns]].to_sql(
            'intl_stats', self.conn, if_exists='append', index=False
        )
        print(f"Loaded {len(intl_df)} International records")

        # Log quality issues
        for issue in quality_issues:
            self.conn.execute(
                "INSERT INTO data_quality_log (issue_description) VALUES (?)",
                (issue,)
            )
        print(f"Logged {len(quality_issues)} data quality issues")

        self.conn.commit()

        # DEBUG: Return filtered DataFrames for pipeline consistency
        print(f"[DEBUG ISSUE #1] Returning filtered DataFrames to pipeline")
        return nba_df, intl_df

    def run_sanity_checks(self) -> 'DatabaseManager':
        """Run SQL queries to verify data integrity."""
        assert self.conn is not None, "Database not connected"

        print("\n" + "=" * 100)
        print("SQL SANITY CHECKS")
        print("=" * 100)

        # Data completeness check
        query = """
        SELECT
            'NBA' as dataset,
            COUNT(*) as total_records,
            SUM(CASE WHEN games IS NULL OR games <= 0 THEN 1 ELSE 0 END) as invalid_games,
            SUM(CASE WHEN minutes IS NULL THEN 1 ELSE 0 END) as missing_minutes,
            SUM(CASE WHEN points IS NULL THEN 1 ELSE 0 END) as missing_points
        FROM nba_stats
        UNION ALL
        SELECT
            'International' as dataset,
            COUNT(*) as total_records,
            SUM(CASE WHEN games IS NULL OR games <= 0 THEN 1 ELSE 0 END) as invalid_games,
            SUM(CASE WHEN minutes IS NULL THEN 1 ELSE 0 END) as missing_minutes,
            SUM(CASE WHEN points IS NULL THEN 1 ELSE 0 END) as missing_points
        FROM intl_stats;
        """
        result = pd.read_sql_query(query, self.conn)
        print("\nData completeness:")
        print(result.to_string(index=False))

        # Top scorers by league
        query = """
        SELECT league,
               SUBSTR(p.first_name, 1, 1) || '. ' || p.last_name as player,
               ROUND(SUM(points) * 1.0 / SUM(games), 1) as ppg,
               SUM(games) as games
        FROM intl_stats i
        JOIN players p ON i.player_id = p.player_id
        WHERE season = 2021
        GROUP BY i.player_id, league
        HAVING SUM(games) >= 10
        ORDER BY league, ppg DESC
        LIMIT 10;
        """
        result = pd.read_sql_query(query, self.conn)
        print("\nTop scorers by league (2021, min 10 games):")
        print(result.to_string(index=False))

        # Quality issues count
        query = "SELECT COUNT(*) as total_issues FROM data_quality_log;"
        result = pd.read_sql_query(query, self.conn)
        print("\nData quality issues logged:")
        print(result.to_string(index=False))

        return self

    def close(self) -> None:
        """Close database connection."""
        if self.conn is not None:
            self.conn.close()
            print(f"\nDatabase connection closed: {self.db_path}")


if __name__ == "__main__":
    """Smoke test with synthetic data and in-memory database."""
    print("=" * 100)
    print("DATABASE MODULE SMOKE TEST")
    print("=" * 100)

    try:
        # Create synthetic test data
        print("\nCreating synthetic test data...")

        # Test player data
        player_df = pd.DataFrame({
            'player_id': ['test_player_1', 'test_player_2'],
            'first_name': ['Test', 'Test'],
            'last_name': ['Player1', 'Player2'],
            'birth_date': ['1995-01-15', '1993-06-20'],
            'birth_year': [1995, 1993],
            'age_2021': [26, 28]
        })

        # Test NBA data
        nba_df = pd.DataFrame({
            'player_id': ['test_player_1'],
            'season': [2021],
            'team': ['TestTeam'],
            'games': [50],
            'minutes': [1200.0],
            'points': [600.0],
            'assists': [200.0],
            'offensive_rebounds': [50.0],
            'defensive_rebounds': [150.0],
            'steals': [30.0],
            'blocked_shots': [20.0],
            'turnovers': [80.0],
            'personal_fouls': [100.0],
            'two_points_made': [150.0],
            'two_points_attempted': [300.0],
            'three_points_made': [50.0],
            'three_points_attempted': [150.0],
            'free_throws_made': [100.0],
            'free_throws_attempted': [120.0],
            'true_shooting_percentage': [0.55],
            'usage_percentage': [0.25],
            'plus_minus': [50.0]
        })

        # Test international data
        intl_df = pd.DataFrame({
            'player_id': ['test_player_2'],
            'season': [2021],
            'league': ['EuroLeague'],
            'team': ['TestTeam'],
            'games': [30],
            'starts': [25],
            'minutes': [800.0],
            'points': [400.0],
            'assists': [120.0],
            'offensive_rebounds': [30.0],
            'defensive_rebounds': [100.0],
            'steals': [20.0],
            'blocked_shots': [15.0],
            'turnovers': [60.0],
            'personal_fouls': [80.0],
            'two_points_made': [100.0],
            'two_points_attempted': [200.0],
            'three_points_made': [40.0],
            'three_points_attempted': [120.0],
            'free_throws_made': [80.0],
            'free_throws_attempted': [100.0],
            'true_shooting_percentage': [0.52],
            'usage_percentage': [0.22]
        })

        # Test quality issues
        quality_issues = ["Test issue 1", "Test issue 2"]

        print("✓ Synthetic data created")

        # Initialize database with in-memory SQLite
        db = DatabaseManager(db_path=':memory:')
        print("\n✓ DatabaseManager initialized")

        # Connect and setup
        db.connect()
        print("✓ Database connected")

        db.create_schema()
        print("✓ Schema created")

        db.create_indexes()
        print("✓ Indexes created")

        # Load data
        db.load_data(player_df, nba_df, intl_df, quality_issues)
        print("✓ Data loaded")

        # Run sanity checks
        db.run_sanity_checks()
        print("✓ Sanity checks passed")

        # Verify table counts
        query = "SELECT COUNT(*) FROM players"
        count = pd.read_sql_query(query, db.conn).iloc[0, 0]
        print(f"\nPlayers in DB: {count}")
        assert count == 2, f"Expected 2 players, got {count}"

        query = "SELECT COUNT(*) FROM nba_stats"
        count = pd.read_sql_query(query, db.conn).iloc[0, 0]
        print(f"NBA records in DB: {count}")
        assert count == 1, f"Expected 1 NBA record, got {count}"

        query = "SELECT COUNT(*) FROM intl_stats"
        count = pd.read_sql_query(query, db.conn).iloc[0, 0]
        print(f"International records in DB: {count}")
        assert count == 1, f"Expected 1 international record, got {count}"

        query = "SELECT COUNT(*) FROM data_quality_log"
        count = pd.read_sql_query(query, db.conn).iloc[0, 0]
        print(f"Quality issues logged: {count}")
        assert count == 2, f"Expected 2 quality issues, got {count}"

        # Close connection
        db.close()
        print("✓ Database closed")

        print("\n" + "=" * 100)
        print("DATABASE MODULE: PASSED ✓")
        print("=" * 100)

    except Exception as e:
        print(f"\n✗ ERROR: {e}")
        import traceback
        traceback.print_exc()
        print("\n" + "=" * 100)
        print("DATABASE MODULE: FAILED ✗")
        print("=" * 100)



Overwriting modular/database/manager.py


In [5]:
%%writefile modular/features/statistics.py
"""
Statistics calculation module - FIXED VERSION
Addresses: Proper counting and documentation of all calculated statistics
"""

import pandas as pd
import numpy as np

from modular.utils import safe_div, calculate_efg_percentage


class StatisticsCalculator:
    """Calculates basketball statistics from raw game data."""
    
    # FIXED: Define all statistics that will be calculated
    CALCULATED_STATS = {
        'per_game': ['ppg', 'apg', 'rpg', 'spg', 'bpg', 'mpg', 'topg', 'fpg'],
        'per_36': ['pts_per_36', 'ast_per_36', 'reb_per_36'],
        'shooting': ['fg_pct', 'three_pt_pct', 'ft_pct', 'efg_pct'],
        'efficiency': ['efficiency'],
        'other': ['usage_percentage', 'plus_minus_per_game']
    }

    @staticmethod
    def get_stat_count():
        """Return total number of calculated statistics."""
        return sum(len(stats) for stats in StatisticsCalculator.CALCULATED_STATS.values())

    @staticmethod
    def get_stat_list():
        """Return flat list of all calculated statistics."""
        all_stats = []
        for stats in StatisticsCalculator.CALCULATED_STATS.values():
            all_stats.extend(stats)
        return all_stats

    @staticmethod
    def calculate_per_game_stats(df: pd.DataFrame) -> pd.DataFrame:
        """Calculate per-game statistics."""
        games = df['games'].replace(0, np.nan)

        df['ppg'] = safe_div(df['points'], games)
        df['apg'] = safe_div(df['assists'], games)
        df['rpg'] = safe_div(
            df['offensive_rebounds'].fillna(0) + df['defensive_rebounds'].fillna(0), 
            games
        )
        df['spg'] = safe_div(df['steals'], games)
        df['bpg'] = safe_div(df['blocked_shots'], games)
        df['mpg'] = safe_div(df['minutes'], games)
        df['topg'] = safe_div(df['turnovers'], games)
        df['fpg'] = safe_div(df.get('personal_fouls', 0), games)

        return df

    @staticmethod
    def calculate_shooting_percentages(df: pd.DataFrame) -> pd.DataFrame:
        """Calculate shooting percentage statistics."""
        df['fg_pct'] = safe_div(df['two_points_made'], df['two_points_attempted'])
        df['three_pt_pct'] = safe_div(df['three_points_made'], df['three_points_attempted'])
        df['ft_pct'] = safe_div(df['free_throws_made'], df['free_throws_attempted'])

        df['efg_pct'] = calculate_efg_percentage(
            df['two_points_made'],
            df['three_points_made'],
            df['two_points_attempted'],
            df['three_points_attempted']
        )

        return df

    @staticmethod
    def calculate_efficiency(df: pd.DataFrame) -> pd.DataFrame:
        """Calculate efficiency rating."""
        games = df['games'].replace(0, np.nan)

        misses = (
            (df['two_points_attempted'].fillna(0) - df['two_points_made'].fillna(0)) +
            (df['three_points_attempted'].fillna(0) - df['three_points_made'].fillna(0)) +
            (df['free_throws_attempted'].fillna(0) - df['free_throws_made'].fillna(0))
        )

        positive = (
            df['points'].fillna(0) +
            df['offensive_rebounds'].fillna(0) + df['defensive_rebounds'].fillna(0) +
            df['assists'].fillna(0) + 
            df['steals'].fillna(0) + 
            df['blocked_shots'].fillna(0)
        )

        negative = misses + df['turnovers'].fillna(0)

        df['efficiency'] = safe_div(positive - negative, games)

        return df

    @staticmethod
    def calculate_per_36_stats(df: pd.DataFrame) -> pd.DataFrame:
        """Calculate per-36-minute statistics."""
        minutes = df['minutes'].replace(0, np.nan)

        df['pts_per_36'] = safe_div(df['points'], minutes) * 36
        df['ast_per_36'] = safe_div(df['assists'], minutes) * 36
        df['reb_per_36'] = safe_div(
            df['offensive_rebounds'].fillna(0) + df['defensive_rebounds'].fillna(0),
            minutes
        ) * 36

        return df

    @staticmethod
    def calculate_usage_percentage(df: pd.DataFrame) -> pd.DataFrame:
        """Calculate usage percentage if not already present."""
        if 'usage_percentage' not in df.columns or df['usage_percentage'].isna().all():
            total_fga = df['two_points_attempted'].fillna(0) + df['three_points_attempted'].fillna(0)

            team_poss = df.get(
                'team_possessions', 
                total_fga + 0.44 * df['free_throws_attempted'].fillna(0) + df['turnovers'].fillna(0)
            )

            player_poss = (
                total_fga + 
                0.44 * df['free_throws_attempted'].fillna(0) + 
                df['turnovers'].fillna(0)
            )

            df['usage_percentage'] = safe_div(player_poss, team_poss)

        return df

    @staticmethod
    def calculate_plus_minus_per_game(df: pd.DataFrame) -> pd.DataFrame:
        """Calculate plus/minus per game if plus_minus column exists."""
        if 'plus_minus' in df.columns:
            games = df['games'].replace(0, np.nan)
            df['plus_minus_per_game'] = safe_div(df['plus_minus'], games)
        else:
            df['plus_minus_per_game'] = np.nan

        return df

    @classmethod
    def calculate_all_statistics(cls, df: pd.DataFrame, dataset_name: str = '') -> pd.DataFrame:
        """Calculate all statistics for a dataset."""
        if dataset_name:
            print(f"Calculating statistics for {dataset_name}...")

        df = cls.calculate_per_game_stats(df)
        df = cls.calculate_shooting_percentages(df)
        df = cls.calculate_efficiency(df)
        df = cls.calculate_per_36_stats(df)
        df = cls.calculate_usage_percentage(df)
        df = cls.calculate_plus_minus_per_game(df)

        return df


def calculate_statistics(nba_df: pd.DataFrame, intl_df: pd.DataFrame) -> tuple:
    """
    Calculate all statistics for NBA and International datasets.
    
    FIXED: Properly reports count of calculated statistics.
    """
    print("\n" + "=" * 100)
    print("CALCULATING STATISTICS")
    print("=" * 100)

    calc = StatisticsCalculator()

    nba_df = calc.calculate_all_statistics(nba_df, 'NBA')
    intl_df = calc.calculate_all_statistics(intl_df, 'International')

    # FIXED: Report accurate count
    stat_count = calc.get_stat_count()
    stat_list = calc.get_stat_list()
    
    print(f"\n✓ Calculated {stat_count} statistics:")
    print(f"  Per-game (8): {', '.join(calc.CALCULATED_STATS['per_game'])}")
    print(f"  Per-36 (3): {', '.join(calc.CALCULATED_STATS['per_36'])}")
    print(f"  Shooting (4): {', '.join(calc.CALCULATED_STATS['shooting'])}")
    print(f"  Efficiency (1): {', '.join(calc.CALCULATED_STATS['efficiency'])}")
    print(f"  Other (2): {', '.join(calc.CALCULATED_STATS['other'])}")

    print("\nStatistics calculation complete.")

    return nba_df, intl_df


if __name__ == "__main__":
    """Smoke test."""
    print("="*100)
    print("STATISTICS MODULE SMOKE TEST (FIXED VERSION)")
    print("="*100)

    try:
        test_data = pd.DataFrame({
            'player_id': ['p1', 'p2'],
            'games': [50, 60],
            'minutes': [1200, 1500],
            'points': [600, 900],
            'assists': [200, 300],
            'offensive_rebounds': [50, 40],
            'defensive_rebounds': [150, 180],
            'steals': [30, 40],
            'blocked_shots': [20, 15],
            'turnovers': [80, 100],
            'personal_fouls': [100, 120],
            'two_points_made': [150, 200],
            'two_points_attempted': [300, 400],
            'three_points_made': [50, 100],
            'three_points_attempted': [150, 300],
            'free_throws_made': [100, 100],
            'free_throws_attempted': [120, 130],
        })

        calc = StatisticsCalculator()
        result = calc.calculate_all_statistics(test_data.copy(), 'Test')

        # Verify stat count
        expected_count = calc.get_stat_count()
        calculated_stats = [col for col in result.columns if col not in test_data.columns]
        actual_count = len(calculated_stats)
        
        print(f"\n✓ Expected {expected_count} stats, calculated {actual_count}")
        assert actual_count == expected_count, f"Stat count mismatch"
        
        # Verify all expected stats present
        expected_stats = calc.get_stat_list()
        for stat in expected_stats:
            assert stat in result.columns, f"Missing {stat}"
        
        print(f"✓ All {expected_count} statistics present in output")
        
        # Verify calculations
        assert abs(result['ppg'].iloc[0] - 12.0) < 0.01, "PPG wrong"
        assert abs(result['apg'].iloc[0] - 4.0) < 0.01, "APG wrong"
        print("✓ Sample calculations verified")
        
        print("\n✓ STATISTICS MODULE (FIXED): PASSED")
        
    except Exception as e:
        print(f"\n✗ ERROR: {e}")
        import traceback
        traceback.print_exc()

Overwriting modular/features/statistics.py


In [6]:
%%writefile modular/analysis/performance.py
"""
Performance analysis module - FIXED VERSION
Addresses: Consistent definitions for trending/improving players with clear thresholds
"""

import pandas as pd
import numpy as np



class PerformanceAnalyzer:
    """Analyzes player performance patterns and development trajectories."""

    @staticmethod
    def calculate_player_trajectories(intl_df: pd.DataFrame, min_seasons: int = 2) -> pd.DataFrame:
        """Calculate player development trajectories from international data."""
        print(f"\nCalculating player trajectories (min {min_seasons} seasons)...")

        intl_sorted = intl_df.sort_values('season')
        intl_grouped = intl_sorted.groupby('player_id')

        first_season = intl_grouped.first()
        last_season = intl_grouped.last()
        season_count = intl_grouped.size()

        traj_df = pd.DataFrame({
            'seasons_played': season_count,
            'ppg_change': last_season['ppg'] - first_season['ppg'],
            'efficiency_change': last_season['efficiency'] - first_season['efficiency'],
            'first_season': first_season['season'],
            'last_season': last_season['season']
        })

        traj_df = traj_df[traj_df['seasons_played'] >= min_seasons]

        print(f"Players with {min_seasons}+ seasons: {len(traj_df):,}")
        if len(traj_df) > 0:
            print(f"Average PPG change: {traj_df['ppg_change'].mean():+.2f}")
            print(f"Average efficiency change: {traj_df['efficiency_change'].mean():+.2f}")

        return traj_df

    @staticmethod
    def identify_trending_players(traj_df: pd.DataFrame, 
                                  ppg_threshold: float = 3.0,
                                  eff_threshold: float = 2.0) -> pd.DataFrame:
        """
        FIXED: Identify trending players with SINGLE consistent definition.
        
        Trending = PPG improved by 3.0+ OR efficiency improved by 2.0+
        """
        trending = traj_df[
            (traj_df['ppg_change'] >= ppg_threshold) |
            (traj_df['efficiency_change'] >= eff_threshold)
        ].copy()

        # Add categorization for reporting
        trending['improvement_category'] = 'Moderate'
        trending.loc[
            (trending['ppg_change'] >= 5.0) | (trending['efficiency_change'] >= 3.0),
            'improvement_category'
        ] = 'Strong'

        moderate = (trending['improvement_category'] == 'Moderate').sum()
        strong = (trending['improvement_category'] == 'Strong').sum()

        print(f"\nTrending players (PPG≥{ppg_threshold} or EFF≥{eff_threshold}): {len(trending)}")
        print(f"  - Moderate improvement: {moderate}")
        print(f"  - Strong improvement: {strong}")

        return trending

    @staticmethod
    def analyze_league_performance(intl_df: pd.DataFrame, 
                                   season: int = 2021,
                                   min_games: int = 10) -> pd.DataFrame:
        """
        FIXED: Analyze performance with consistent game filter.
        """
        season_data = intl_df[
            (intl_df['season'] == season) & 
            (intl_df['games'] >= min_games)
        ].copy()

        if len(season_data) == 0:
            print(f"\nNo data for season {season} with min {min_games} games")
            return pd.DataFrame()

        league_stats = season_data.groupby('league').agg({
            'ppg': ['mean', 'median', 'std'],
            'apg': ['mean', 'median'],
            'rpg': ['mean', 'median'],
            'efficiency': ['mean', 'median'],
            'three_pt_pct': ['mean', 'median'],
            'player_id': 'count'
        }).round(2)

        league_stats.columns = ['_'.join(col).strip() for col in league_stats.columns.values]
        league_stats = league_stats.rename(columns={'player_id_count': 'num_players'})

        total_players = league_stats['num_players'].sum()
        print(f"\nLeague performance summary ({season}, min {min_games} games):")
        print(f"  Leagues analyzed: {len(league_stats)}")
        print(f"  Total players: {int(total_players)}")

        return league_stats.reset_index()

    @staticmethod
    def compare_nba_vs_international(nba_df: pd.DataFrame, 
                                    intl_df: pd.DataFrame,
                                    season: int = 2021,
                                    min_games: int = 10) -> dict:
        """
        FIXED: Compare with consistent filters applied to both datasets.
        """
        # Apply same filter to both
        nba_season = nba_df[(nba_df['season'] == season) & (nba_df['games'] >= min_games)]
        intl_season = intl_df[(intl_df['season'] == season) & (intl_df['games'] >= min_games)]

        comparison = {
            'season': season,
            'min_games': min_games,
            'nba_players': len(nba_season['player_id'].unique()),
            'intl_players': len(intl_season['player_id'].unique()),
            'nba_ppg_mean': nba_season['ppg'].mean(),
            'intl_ppg_mean': intl_season['ppg'].mean(),
            'nba_efficiency_mean': nba_season['efficiency'].mean(),
            'intl_efficiency_mean': intl_season['efficiency'].mean(),
            'nba_3pt_pct_mean': nba_season['three_pt_pct'].mean(),
            'intl_3pt_pct_mean': intl_season['three_pt_pct'].mean(),
        }

        print(f"\nNBA vs International comparison ({season}, min {min_games} games):")
        print(f"  NBA: {comparison['nba_players']} players")
        print(f"  International: {comparison['intl_players']} players")
        print(f"  Total: {comparison['nba_players'] + comparison['intl_players']} players")
        print(f"  PPG - NBA: {comparison['nba_ppg_mean']:.1f}, Intl: {comparison['intl_ppg_mean']:.1f}")
        print(f"  EFF - NBA: {comparison['nba_efficiency_mean']:.1f}, Intl: {comparison['intl_efficiency_mean']:.1f}")

        return comparison


def analyze_performance_patterns(intl_df: pd.DataFrame, 
                                nba_df: pd.DataFrame = None) -> pd.DataFrame:
    """
    Main function to analyze performance patterns.
    
    FIXED: Consistent filtering and clear definitions throughout.
    """
    print("\n" + "=" * 100)
    print("ANALYZING PERFORMANCE PATTERNS")
    print("=" * 100)

    analyzer = PerformanceAnalyzer()

    traj_df = analyzer.calculate_player_trajectories(intl_df, min_seasons=2)

    if len(traj_df) > 0:
        trending = analyzer.identify_trending_players(traj_df)

    # FIXED: Use consistent min_games threshold
    league_stats = analyzer.analyze_league_performance(intl_df, season=2021, min_games=10)

    if nba_df is not None:
        # FIXED: Apply same filter to both datasets
        comparison = analyzer.compare_nba_vs_international(nba_df, intl_df, season=2021, min_games=10)

    print("\nPerformance analysis complete.")

    return traj_df


if __name__ == "__main__":
    """Smoke test."""
    print("="*100)
    print("ANALYSIS MODULE SMOKE TEST (FIXED VERSION)")
    print("="*100)

    try:
        test_intl = pd.DataFrame({
            'player_id': ['p1', 'p1', 'p1', 'p2', 'p2', 'p3'],
            'season': [2019, 2020, 2021, 2020, 2021, 2021],
            'league': ['EuroLeague', 'EuroLeague', 'EuroLeague', 'ACB', 'ACB', 'EuroLeague'],
            'games': [30, 35, 40, 25, 30, 20],
            'ppg': [10, 12, 15, 8, 9, 12],
            'apg': [3, 4, 5, 2, 3, 4],
            'rpg': [5, 5, 6, 4, 4, 5],
            'efficiency': [8, 10, 12, 6, 7, 9],
            'three_pt_pct': [0.35, 0.37, 0.40, 0.30, 0.32, 0.38]
        })

        test_nba = pd.DataFrame({
            'player_id': ['n1', 'n2'],
            'season': [2021, 2021],
            'games': [50, 60],
            'ppg': [15, 18],
            'efficiency': [12, 14],
            'three_pt_pct': [0.38, 0.40]
        })

        analyzer = PerformanceAnalyzer()
        
        # Test trajectories
        traj = analyzer.calculate_player_trajectories(test_intl, min_seasons=2)
        assert len(traj) > 0, "No trajectories calculated"
        print(f"\n✓ Trajectories: {len(traj)} players")

        # Test trending with clear definition
        trending = analyzer.identify_trending_players(traj, ppg_threshold=3.0)
        print(f"✓ Trending players: {len(trending)}")
        
        # Verify categories exist
        if len(trending) > 0:
            assert 'improvement_category' in trending.columns, "Missing improvement category"
            print(f"✓ Improvement categories assigned")

        # Test league analysis with filter
        league_stats = analyzer.analyze_league_performance(test_intl, season=2021, min_games=10)
        print(f"✓ League analysis: {len(league_stats)} leagues")

        # Test comparison with consistent filters
        comparison = analyzer.compare_nba_vs_international(test_nba, test_intl, season=2021, min_games=10)
        assert 'min_games' in comparison, "Missing filter documentation"
        total = comparison['nba_players'] + comparison['intl_players']
        print(f"✓ Comparison: {total} total players (consistent filter)")

        print("\n✓ ANALYSIS MODULE (FIXED): PASSED")

    except Exception as e:
        print(f"\n✗ ERROR: {e}")
        import traceback
        traceback.print_exc()

Overwriting modular/analysis/performance.py


In [7]:
%%writefile modular/models/predictor.py
"""
Machine learning module for NBA success prediction.
Trains a calibrated classifier to predict NBA success from international stats.
"""

import pandas as pd
import numpy as np
from typing import Dict, Optional


from sklearn.ensemble import GradientBoostingClassifier
from sklearn.model_selection import cross_val_score, cross_validate, train_test_split
from sklearn.metrics import (
    roc_auc_score, classification_report, precision_recall_curve,
    average_precision_score, brier_score_loss, roc_curve
)
from sklearn.calibration import CalibratedClassifierCV, calibration_curve
from sklearn.preprocessing import StandardScaler
from sklearn.inspection import permutation_importance
SKLEARN_AVAILABLE = True


# When run as part of package
from modular.config import Config
from modular.utils import safe_div



class NBASuccessPredictor:
    """Predicts NBA success probability from international performance."""

    def __init__(self):
        """Initialize the predictor."""
        self.ml_artifacts = {
            'enabled': False,
            'model': None,
            'scaler': None,
            'features': None,
            'metrics': None,
            'feature_importance': None
        }
        self.current_year_predictions = None

    def define_nba_success(self, nba_df: pd.DataFrame, min_games: int = 10) -> pd.Series:
        """
        Define NBA success criteria.

        Success = MPG >= 15 AND (PPG >= 8 OR (PPG >= 5 AND TS% >= 0.58))

        Args:
            nba_df: NBA statistics DataFrame
            min_games: Minimum games threshold

        Returns:
            Series of player_id -> success boolean
        """
        # Filter to meaningful NBA seasons
        nba_filtered = nba_df[nba_df['games'] >= min_games].copy()

        # Calculate success per season
        nba_filtered['success'] = (
            (nba_filtered['mpg'] >= Config.NBA_SUCCESS_MIN_MPG) &
            (
                (nba_filtered['ppg'] >= Config.NBA_SUCCESS_MIN_PPG) |
                (
                    (nba_filtered['ppg'] >= Config.NBA_SUCCESS_MIN_PPG_ALT) &
                    (nba_filtered['true_shooting_percentage'] >= Config.NBA_SUCCESS_MIN_TS)
                )
            )
        )

        # A player is successful if they had ANY successful season
        player_success = nba_filtered.groupby('player_id')['success'].max()

        return player_success

    def extract_features(self, player_id: str, intl_df: pd.DataFrame, 
                        first_nba_season: int) -> Optional[Dict]:
        """
        Extract features from last international season before NBA.

        Args:
            player_id: Player identifier
            intl_df: International statistics DataFrame
            first_nba_season: First NBA season for this player

        Returns:
            Dictionary of features or None if insufficient data
        """
        # Get international history before NBA
        intl_hist = intl_df[
            (intl_df['player_id'] == player_id) & 
            (intl_df['season'] < first_nba_season)
        ].copy()

        if intl_hist.empty:
            return None

        # Get last season stats
        intl_last = intl_hist.sort_values('season').iloc[-1]

        # Calculate trajectory features
        ppg_trend = 0
        if len(intl_hist) >= 2:
            ppg_trend = intl_hist.iloc[-1]['ppg'] - intl_hist.iloc[0]['ppg']

        # Build feature dictionary
        features = {
            'ppg': float(intl_last.get('ppg', 0) or 0),
            'apg': float(intl_last.get('apg', 0) or 0),
            'rpg': float(intl_last.get('rpg', 0) or 0),
            'spg': float(intl_last.get('spg', 0) or 0),
            'bpg': float(intl_last.get('bpg', 0) or 0),
            'mpg': float(intl_last.get('mpg', 0) or 0),
            'three_pt_pct': float(intl_last.get('three_pt_pct', 0) or 0),
            'ft_pct': float(intl_last.get('ft_pct', 0) or 0),
            'efficiency': float(intl_last.get('efficiency', 0) or 0),
            'ts_pct': float(intl_last.get('true_shooting_percentage', 0) or 0),
            'ppg_trend': ppg_trend,
            'seasons_played': len(intl_hist),
            'ast_to_tov': float(intl_last.get('apg', 0) / max(intl_last.get('topg', 1), 0.1)),
            'pts_per_min': float(intl_last.get('ppg', 0) / max(intl_last.get('mpg', 1), 1)),
            'usage_pct': float(intl_last.get('usage_percentage', 0) or 0),
            'efg_pct': float(intl_last.get('efg_pct', 0) or 0),
            'games': float(intl_last.get('games', 0) or 0),
        }

        return features

    def build_training_data(self, player_df: pd.DataFrame, nba_df: pd.DataFrame,
                           intl_df: pd.DataFrame) -> tuple:
        """
        Build training dataset from players who played in both leagues.

        Args:
            player_df: Player demographics DataFrame
            nba_df: NBA statistics DataFrame
            intl_df: International statistics DataFrame

        Returns:
            Tuple of (X, y, player_ids) or (None, None, None) if insufficient data
        """
        # Find players with both NBA and international experience
        both_leagues = set(nba_df['player_id']).intersection(set(intl_df['player_id']))

        if len(both_leagues) < 50:
            print(f"Insufficient overlap ({len(both_leagues)} players); need at least 50")
            return None, None, None

        print(f"\n[DEBUG ISSUE #3] Training data construction:")
        print(f"  Players with both NBA + Intl: {len(both_leagues)}")

        # Define success
        player_success = self.define_nba_success(nba_df)

        print(f"  Successful players: {player_success.sum()} ({player_success.mean():.1%})")
        print(f"  Failed players: {(~player_success).sum()}")

        # Build training examples
        rows = []
        leakage_count = 0
        leakage_samples = []

        for pid in both_leagues:
            # Get first NBA season
            nba_seasons = sorted(nba_df[nba_df['player_id'] == pid]['season'].unique())
            if not len(nba_seasons):
                continue

            first_nba = nba_seasons[0]

            # DEBUG ISSUE #3: Check for temporal leakage
            intl_seasons = sorted(intl_df[intl_df['player_id'] == pid]['season'].unique())
            if intl_seasons and intl_seasons[-1] >= first_nba:
                leakage_count += 1
                if len(leakage_samples) < 5:
                    leakage_samples.append({
                        'player_id': pid,
                        'first_nba': first_nba,
                        'last_intl': intl_seasons[-1]
                    })

            # Extract features
            features = self.extract_features(pid, intl_df, first_nba)
            if features is None:
                continue

            # Get success label
            success = player_success.get(pid, False)

            rows.append((pid, features, int(success)))

        # DEBUG ISSUE #3: Report temporal leakage
        if leakage_count > 0:
            print(f"\n[DEBUG ISSUE #3] ⚠️ TEMPORAL LEAKAGE DETECTED:")
            print(f"  {leakage_count} players have Intl data >= their first NBA season")
            print(f"  Sample cases:")
            for sample in leakage_samples:
                print(f"    {sample['player_id']}: NBA start {sample['first_nba']}, Intl last {sample['last_intl']}")

        if len(rows) < 50:
            print(f"Insufficient training examples ({len(rows)}); need at least 50")
            return None, None, None

        # Convert to DataFrames
        player_ids = [r[0] for r in rows]
        X = pd.DataFrame([r[1] for r in rows])
        y = pd.Series([r[2] for r in rows])

        # Handle missing/invalid values
        X = X.fillna(0).replace([np.inf, -np.inf], 0)

        # DEBUG ISSUE #3: Feature distribution analysis
        print(f"\n[DEBUG ISSUE #3] Feature distributions:")
        for col in ['ppg', 'efficiency', 'three_pt_pct', 'ppg_trend', 'mpg']:
            if col in X.columns:
                print(f"  {col}: mean={X[col].mean():.2f}, std={X[col].std():.2f}, "
                      f"min={X[col].min():.2f}, max={X[col].max():.2f}")

        # DEBUG ISSUE #3: Class separation analysis
        print(f"\n[DEBUG ISSUE #3] Feature means by outcome class:")
        X_with_y = X.copy()
        X_with_y['target'] = y
        for col in ['ppg', 'efficiency', 'three_pt_pct', 'mpg']:
            if col in X.columns:
                mean_success = X_with_y[X_with_y['target']==1][col].mean()
                mean_fail = X_with_y[X_with_y['target']==0][col].mean()
                separation = mean_success - mean_fail
                print(f"  {col}: Success={mean_success:.2f}, Failure={mean_fail:.2f}, Δ={separation:.2f}")

        return X, y, player_ids

    def train_model(self, X: pd.DataFrame, y: pd.Series) -> bool:
        """
        Train calibrated gradient boosting classifier.

        Args:
            X: Feature matrix
            y: Target labels

        Returns:
            True if training successful, False otherwise
        """
        if not SKLEARN_AVAILABLE:
            print("scikit-learn not available; skipping model training")
            return False

        print(f"\nTraining model on {len(X)} examples...")
        print(f"  Positive class: {y.sum()} ({y.mean()*100:.1f}%)")
        print(f"  Features: {len(X.columns)}")

        # Train/test split
        X_train, X_test, y_train, y_test = train_test_split(
            X, y, 
            test_size=Config.ML_TEST_SIZE,
            stratify=y,
            random_state=Config.ML_RANDOM_STATE
        )

        # Scale features
        scaler = StandardScaler()
        X_train_scaled = scaler.fit_transform(X_train)
        X_test_scaled = scaler.transform(X_test)

        # Train base model
        base_clf = GradientBoostingClassifier(
            n_estimators=Config.ML_N_ESTIMATORS,
            max_depth=Config.ML_MAX_DEPTH,
            learning_rate=Config.ML_LEARNING_RATE,
            random_state=Config.ML_RANDOM_STATE
        )
        base_clf.fit(X_train_scaled, y_train)

        # Calibrate probabilities
        clf = CalibratedClassifierCV(base_clf, method='sigmoid', cv='prefit')
        clf.fit(X_train_scaled, y_train)

        # Evaluate
        y_prob_uncal = base_clf.predict_proba(X_test_scaled)[:, 1]
        y_prob = clf.predict_proba(X_test_scaled)[:, 1]

        # Calculate metrics
        metrics = self._calculate_metrics(y_test, y_prob, y_prob_uncal, X_train_scaled, y_train, base_clf)

        # Feature importance
        feat_imp = self._calculate_feature_importance(
            base_clf, clf, X, X_test_scaled, y_test
        )

        # Store artifacts
        self.ml_artifacts = {
            'enabled': True,
            'model': clf,
            'base_model': base_clf,
            'scaler': scaler,
            'features': list(X.columns),
            'metrics': metrics,
            'feature_importance': feat_imp
        }

        print(f"\n✓ Model trained successfully")
        print(f"  ROC-AUC: {metrics['test_auc']:.3f}")
        print(f"  PR-AUC: {metrics['test_pr_auc']:.3f}")
        print(f"  Brier Score (calibrated): {metrics['brier_score_calibrated']:.3f}")

        # Save feature importance
        feat_imp.to_csv('feature_importance.csv', index=False)
        print(f"  Saved: feature_importance.csv")

        return True

    def _calculate_metrics(self, y_test, y_prob, y_prob_uncal, 
                          X_train_scaled, y_train, base_clf) -> Dict:
        """Calculate comprehensive model metrics."""
        # Basic metrics
        test_auc = roc_auc_score(y_test, y_prob)
        pr_auc = average_precision_score(y_test, y_prob)
        brier = brier_score_loss(y_test, y_prob)
        brier_uncal = brier_score_loss(y_test, y_prob_uncal)

        # Precision at K
        sorted_indices = np.argsort(y_prob)[::-1]
        precision_at = {}
        for k in [10, 20, 30]:
            if k <= len(y_test):
                precision_at[k] = y_test.iloc[sorted_indices[:k]].mean()

        # Cross-validation
        cv_results = cross_validate(
            base_clf, X_train_scaled, y_train,
            cv=Config.ML_CV_FOLDS,
            scoring=['roc_auc', 'average_precision'],
            return_train_score=True
        )

        # Bootstrap confidence intervals
        np.random.seed(Config.ML_RANDOM_STATE)
        bootstrap_aucs = []
        for _ in range(Config.ML_N_BOOTSTRAP):
            indices = np.random.choice(len(y_test), size=len(y_test), replace=True)
            if len(np.unique(y_test.iloc[indices])) < 2:
                continue
            bootstrap_aucs.append(roc_auc_score(y_test.iloc[indices], y_prob[indices]))

        auc_ci = np.percentile(bootstrap_aucs, [2.5, 97.5]) if bootstrap_aucs else [test_auc, test_auc]

        return {
            'test_auc': float(test_auc),
            'test_auc_ci_low': float(auc_ci[0]),
            'test_auc_ci_high': float(auc_ci[1]),
            'test_pr_auc': float(pr_auc),
            'cv_auc_median': float(np.median(cv_results['test_roc_auc'])),
            'cv_pr_auc_median': float(np.median(cv_results['test_average_precision'])),
            'brier_score_calibrated': float(brier),
            'brier_score_uncalibrated': float(brier_uncal),
            'brier_improvement': float(brier_uncal - brier),
            'precision_at_10': float(precision_at.get(10, np.nan)),
            'precision_at_20': float(precision_at.get(20, np.nan)),
            'precision_at_30': float(precision_at.get(30, np.nan)),
            'n_train': int(len(X_train_scaled)),
            'n_test': int(len(y_test))
        }

    def _calculate_feature_importance(self, base_clf, clf, X, X_test_scaled, y_test) -> pd.DataFrame:
        """Calculate feature importance using multiple methods."""
        # Gini importance from base model
        gini_imp = base_clf.feature_importances_

        # Permutation importance
        perm_imp = permutation_importance(
            clf, X_test_scaled, y_test,
            n_repeats=Config.ML_PERM_IMPORTANCE_REPEATS,
            random_state=Config.ML_RANDOM_STATE,
            scoring='average_precision'
        )

        feat_imp = pd.DataFrame({
            'feature': X.columns,
            'importance_gini': gini_imp,
            'importance_perm': perm_imp.importances_mean,
            'importance_perm_std': perm_imp.importances_std
        }).sort_values('importance_perm', ascending=False)

        return feat_imp

    def predict_current_prospects(self, intl_df: pd.DataFrame, 
                                 season: int = 2021) -> pd.DataFrame:
        """
        Generate predictions for current international prospects.

        Args:
            intl_df: International statistics DataFrame
            season: Season to predict for

        Returns:
            DataFrame with player_id and nba_success_prob
        """
        if not self.ml_artifacts['enabled']:
            return pd.DataFrame(columns=['player_id', 'nba_success_prob'])

        current_intl = intl_df[intl_df['season'] == season].copy()
        if current_intl.empty:
            return pd.DataFrame(columns=['player_id', 'nba_success_prob'])

        # Extract features for current season
        feats_current = pd.DataFrame()
        for feat in self.ml_artifacts['features']:
            if feat in current_intl.columns:
                feats_current[feat] = current_intl[feat].fillna(0)
            elif feat == 'ppg_trend':
                # Calculate trend for each player
                trends = []
                for pid in current_intl['player_id']:
                    hist = intl_df[intl_df['player_id'] == pid].sort_values('season')
                    if len(hist) >= 2:
                        trends.append(hist.iloc[-1]['ppg'] - hist.iloc[0]['ppg'])
                    else:
                        trends.append(0)
                feats_current[feat] = trends
            elif feat == 'ast_to_tov':
                feats_current[feat] = current_intl['apg'] / current_intl['topg'].replace(0, 0.1)
            elif feat == 'pts_per_min':
                feats_current[feat] = current_intl['ppg'] / current_intl['mpg'].replace(0, 1)
            else:
                feats_current[feat] = 0

        # Ensure correct column order
        feats_current = feats_current[self.ml_artifacts['features']]
        feats_current = feats_current.fillna(0).replace([np.inf, -np.inf], 0)

        # Scale and predict
        feats_scaled = self.ml_artifacts['scaler'].transform(feats_current)
        probs = self.ml_artifacts['model'].predict_proba(feats_scaled)[:, 1]

        result = pd.DataFrame({
            'player_id': current_intl['player_id'],
            'nba_success_prob': probs
        })

        self.current_year_predictions = result
        print(f"\n✓ Generated predictions for {len(result)} current prospects")

        return result

    def get_artifacts(self) -> Dict:
        """Get ML artifacts for use in downstream modules."""
        return self.ml_artifacts


def train_nba_success_model(player_df: pd.DataFrame, nba_df: pd.DataFrame, 
                            intl_df: pd.DataFrame) -> Dict:
    """
    Main function to train NBA success prediction model.

    Args:
        player_df: Player demographics DataFrame
        nba_df: NBA statistics DataFrame  
        intl_df: International statistics DataFrame

    Returns:
        Dictionary of ML artifacts
    """
    print("\n" + "=" * 100)
    print("TRAINING NBA SUCCESS MODEL")
    print("=" * 100)

    if not SKLEARN_AVAILABLE:
        print("\nscikit-learn not available; skipping model training")
        return {'enabled': False}

    predictor = NBASuccessPredictor()

    # Build training data
    X, y, player_ids = predictor.build_training_data(player_df, nba_df, intl_df)

    if X is None:
        print("\nInsufficient data for model training")
        return {'enabled': False}

    # Train model
    success = predictor.train_model(X, y)

    if not success:
        return {'enabled': False}

    # Generate predictions for current prospects
    predictor.predict_current_prospects(intl_df, season=Config.CURRENT_SEASON)

    return predictor.get_artifacts()


Overwriting modular/models/predictor.py


In [8]:
%%writefile modular/scouting/targets.py
"""
Scouting target identification module - FIXED VERSION
Addresses: Always document ml_multiplier value, even when ML disabled
"""

import pandas as pd
import numpy as np
from typing import Dict, Optional

from modular.config import Config



class ScoutingAnalyzer:
    """Analyzes and ranks international prospects for NBA scouting."""

    def __init__(self, team_weights: Optional[Dict] = None, nba_prob_weight: float = 1.10):
        self.team_weights = team_weights or Config.DEFAULT_TEAM_WEIGHTS
        self.nba_prob_weight = nba_prob_weight

    def calculate_team_weights(self, nba_df: pd.DataFrame) -> 'ScoutingAnalyzer':
        """Calculate team fit weights based on successful NBA players."""
        print("\nCalculating team fit weights...")
        print("Definition: NBA seasons with MPG ≥ 20. Weights reflect prevalence above ~75th percentile.")

        successful_nba = nba_df[nba_df['mpg'] >= 20].copy()

        if len(successful_nba) < 50:
            print("Not enough NBA data; using conservative defaults.")
            self.team_weights = Config.DEFAULT_TEAM_WEIGHTS
        else:
            p75_3pt = successful_nba['three_pt_pct'].quantile(0.75)
            p75_def = (successful_nba['spg'] + successful_nba['bpg']).quantile(0.75)
            p75_ast = successful_nba['apg'].quantile(0.75)
            p75_reb = successful_nba['rpg'].quantile(0.75)

            high_3pt = (successful_nba['three_pt_pct'] >= p75_3pt).sum()
            high_def = ((successful_nba['spg'] + successful_nba['bpg']) >= p75_def).sum()
            high_ast = (successful_nba['apg'] >= p75_ast).sum()
            high_reb = (successful_nba['rpg'] >= p75_reb).sum()

            total = len(successful_nba)
            self.team_weights = {
                'shooting_3pt': 1.0 + (high_3pt / total) * 0.5,
                'defense': 1.0 + (high_def / total) * 0.4,
                'playmaking': 1.0 + (high_ast / total) * 0.4,
                'rebounding': 1.0 + (high_reb / total) * 0.3,
                'youth': 1.05
            }

        print("Team fit weights:")
        for k, v in self.team_weights.items():
            print(f"  {k}: {v:.3f}")
        return self

    def normalize_metrics(self, df: pd.DataFrame, metrics: list) -> pd.DataFrame:
        """Normalize metrics to 0-100 scale."""
        def _norm(s):
            s = s.astype(float)
            mn, mx = s.min(), s.max()
            if pd.isna(mn) or pd.isna(mx) or mx <= mn:
                return pd.Series(np.nan, index=s.index)
            return (s - mn) / (mx - mn) * 100.0

        for metric in metrics:
            if metric in df.columns:
                df[f'{metric}_norm'] = _norm(df[metric])
        return df

    def calculate_performance_score(self, prospects: pd.DataFrame) -> pd.DataFrame:
        """Calculate weighted performance score."""
        metrics = ['ppg', 'apg', 'rpg', 'efficiency', 'true_shooting_percentage']
        prospects = self.normalize_metrics(prospects, metrics)

        prospects['performance_score'] = (
            prospects.get('ppg_norm', np.nan).fillna(50) * 0.30 +
            prospects.get('efficiency_norm', np.nan).fillna(50) * 0.25 +
            prospects.get('true_shooting_percentage_norm', np.nan).fillna(50) * 0.20 +
            prospects.get('apg_norm', np.nan).fillna(50) * 0.15 +
            prospects.get('rpg_norm', np.nan).fillna(50) * 0.10
        )

        return prospects

    def calculate_age_bonus(self, prospects: pd.DataFrame) -> pd.DataFrame:
        """Calculate age-based bonus multiplier."""
        prospects['age_bonus'] = prospects['age_2021'].apply(
            lambda x: 1.3 if x < 24 else (1.2 if x < 26 else (1.1 if x < 28 else 1.0))
        )
        return prospects

    def calculate_improvement_bonus(self, prospects: pd.DataFrame) -> pd.DataFrame:
        """Calculate improvement trend bonus."""
        prospects['improvement_bonus'] = prospects['ppg_change'].fillna(0).apply(
            lambda x: 1.2 if x > 5 else (1.15 if x > 3 else (1.1 if x > 1 else 1.0))
        )
        return prospects

    def calculate_team_fit_multiplier(self, prospects: pd.DataFrame) -> pd.DataFrame:
        """Calculate team fit multiplier based on skill components."""
        prospects['def_events_pg'] = (
            prospects.get('spg', 0).fillna(0) + prospects.get('bpg', 0).fillna(0)
        )

        fit_components = {
            'shooting_3pt': self.normalize_metrics(prospects.copy(), ['three_pt_pct'])['three_pt_pct_norm'],
            'defense': self.normalize_metrics(prospects.copy(), ['def_events_pg'])['def_events_pg_norm'],
            'playmaking': self.normalize_metrics(prospects.copy(), ['apg'])['apg_norm'],
            'rebounding': self.normalize_metrics(prospects.copy(), ['rpg'])['rpg_norm']
        }

        prospects['fit_multiplier'] = (
            (1.0 + (fit_components['shooting_3pt'].fillna(50)/100) * (self.team_weights.get('shooting_3pt', 1.0) - 1.0)) *
            (1.0 + (fit_components['defense'].fillna(50)/100) * (self.team_weights.get('defense', 1.0) - 1.0)) *
            (1.0 + (fit_components['playmaking'].fillna(50)/100) * (self.team_weights.get('playmaking', 1.0) - 1.0)) *
            (1.0 + (fit_components['rebounding'].fillna(50)/100) * (self.team_weights.get('rebounding', 1.0) - 1.0)) *
            (prospects['age_bonus'] ** (self.team_weights.get('youth', 1.0) - 1.0))
        )

        return prospects

    def calculate_ml_multiplier(self, prospects: pd.DataFrame) -> pd.DataFrame:
        """
        FIXED: Calculate ML prediction multiplier with explicit documentation.
        """
        # FIXED: Always initialize to 1.0 and document
        prospects['ml_multiplier'] = 1.0

        if 'nba_success_prob' in prospects.columns and prospects['nba_success_prob'].notna().any():
            prospects['ml_multiplier'] = (
                1.0 + (prospects['nba_success_prob'].fillna(0.5) - 0.5) * 
                (self.nba_prob_weight - 1.0) * 2.0
            )
            print(f"NBA success probability weight: {self.nba_prob_weight:.2f}x")
        else:
            # FIXED: Explicitly document when ML not used
            print(f"ML predictions not available; ml_multiplier = 1.0 for all prospects")

        return prospects

    def calculate_scout_score(self, prospects: pd.DataFrame) -> pd.DataFrame:
        """Calculate final scout score with formula documentation."""
        print("\nScout score formula:")
        print("  performance_score × age_bonus × improvement_bonus × fit_multiplier × ml_multiplier")

        prospects['scout_score'] = (
            prospects['performance_score'] *
            prospects['age_bonus'] *
            prospects['improvement_bonus'] *
            prospects['fit_multiplier'] *
            prospects['ml_multiplier']
        )

        # DEBUG ISSUE #4: Decompose top 5 scout scores
        print(f"\n[DEBUG ISSUE #4] Top 5 scout score decomposition:")
        top5 = prospects.nlargest(5, 'scout_score')

        for rank, (idx, row) in enumerate(top5.iterrows(), 1):
            name = f"{row.get('first_name', 'Unknown')} {row.get('last_name', 'Unknown')}"
            print(f"\nRank #{rank}: {name}")
            print(f"  Final scout_score: {row['scout_score']:.1f}")
            print(f"  Components:")
            print(f"    performance_score: {row.get('performance_score', np.nan):.1f}")
            print(f"    age_bonus: {row.get('age_bonus', 1.0):.3f} (age {int(row.get('age_2021', 0))})")
            print(f"    improvement_bonus: {row.get('improvement_bonus', 1.0):.3f} (Δppg {row.get('ppg_change', 0):.1f})")
            print(f"    fit_multiplier: {row.get('fit_multiplier', 1.0):.3f}")
            print(f"    ml_multiplier: {row.get('ml_multiplier', 1.0):.3f}")

            # Show ML probability if available
            if 'nba_success_prob' in row and pd.notna(row['nba_success_prob']):
                print(f"    nba_success_prob: {row['nba_success_prob']:.1%}")
            else:
                print(f"    nba_success_prob: N/A")

            # Verify calculation
            calculated = (row['performance_score'] * row['age_bonus'] *
                         row['improvement_bonus'] * row['fit_multiplier'] *
                         row['ml_multiplier'])
            diff = abs(calculated - row['scout_score'])
            if diff > 0.1:
                print(f"  ⚠️ CALCULATION MISMATCH: Expected {calculated:.1f}, got {row['scout_score']:.1f}")

        return prospects

    def identify_scouting_targets(self, player_df: pd.DataFrame, 
                                nba_df: pd.DataFrame,
                                intl_df: pd.DataFrame,
                                traj_df: Optional[pd.DataFrame] = None,
                                ml_predictions: Optional[pd.DataFrame] = None,
                                season: int = 2021,
                                min_games: int = 10,
                                min_mpg: int = 20,
                                max_age: int = 30,
                                top_n: int = 30) -> pd.DataFrame:
        """Identify and rank scouting targets."""
        print(f"\nSCOUTING TARGETS")
        self.calculate_team_weights(nba_df)

        prospects = intl_df[intl_df['season'] == season].copy()
        prospects = prospects.merge(
            player_df[['player_id', 'age_2021']], on='player_id', how='left'
        )

        print(f"Criteria: season={season}, games≥{min_games}, mpg≥{min_mpg}, age<{max_age}")
        prospects = prospects[
            (prospects['games'] >= min_games) &
            (prospects['mpg'] >= min_mpg) &
            (prospects['age_2021'] < max_age)
        ].copy()
        print(f"Eligible players: {len(prospects)}")

        prospects['has_nba_exp'] = prospects['player_id'].isin(nba_df['player_id'].unique())

        if traj_df is not None:
            traj_merge = traj_df[['ppg_change', 'efficiency_change', 'seasons_played']].reset_index()
            prospects = prospects.merge(traj_merge, on='player_id', how='left')
        else:
            prospects['ppg_change'] = np.nan
            prospects['efficiency_change'] = np.nan

        if ml_predictions is not None:
            prospects = prospects.merge(ml_predictions, on='player_id', how='left')
        else:
            prospects['nba_success_prob'] = np.nan

        prospects = self.calculate_performance_score(prospects)
        prospects = self.calculate_age_bonus(prospects)
        prospects = self.calculate_improvement_bonus(prospects)
        prospects = self.calculate_team_fit_multiplier(prospects)
        prospects = self.calculate_ml_multiplier(prospects)  # FIXED: Always documents value
        prospects = self.calculate_scout_score(prospects)

        prospects = prospects.sort_values('scout_score', ascending=False).drop_duplicates(
            subset='player_id', keep='first'
        )
        top_prospects = prospects.nlargest(top_n, 'scout_score').copy()

        print(f"\nTop {top_n} prospects identified.")
        print(f"With NBA experience: {int(top_prospects['has_nba_exp'].sum())}")
        print(f"International only: {int((~top_prospects['has_nba_exp']).sum())}")

        return top_prospects


def identify_scouting_targets(player_df: pd.DataFrame, 
                            nba_df: pd.DataFrame,
                            intl_df: pd.DataFrame,
                            traj_df: Optional[pd.DataFrame] = None,
                            ml_predictions: Optional[pd.DataFrame] = None,
                            team_weights: Optional[Dict] = None,
                            nba_prob_weight: float = 1.10,
                            season: int = 2021,
                            top_n: int = 30) -> pd.DataFrame:
    """Main function to identify scouting targets."""
    print("\n" + "=" * 100)
    print("IDENTIFYING SCOUTING TARGETS")
    print("=" * 100)

    analyzer = ScoutingAnalyzer(team_weights, nba_prob_weight)

    prospects = analyzer.identify_scouting_targets(
        player_df, nba_df, intl_df, traj_df, ml_predictions, season, top_n=top_n
    )

    print("\nScouting target identification complete.")
    return prospects


if __name__ == "__main__":
    """Smoke test."""
    print("="*100)
    print("SCOUTING MODULE SMOKE TEST (FIXED VERSION)")
    print("="*100)

    try:
        test_player_df = pd.DataFrame({
            'player_id': ['p1', 'p2', 'p3'],
            'age_2021': [22, 25, 28]
        })

        test_nba_df = pd.DataFrame({
            'player_id': ['p1', 'p2'],
            'season': [2021, 2021],
            'mpg': [25, 22],
            'ppg': [15, 12],
            'apg': [5, 4],
            'rpg': [6, 5],
            'spg': [1.5, 1.2],
            'bpg': [0.8, 0.6],
            'three_pt_pct': [0.35, 0.32]
        })

        test_intl_df = pd.DataFrame({
            'player_id': ['p1', 'p2', 'p3'],
            'season': [2021, 2021, 2021],
            'league': ['EuroLeague', 'ACB', 'EuroLeague'],
            'team': ['TeamA', 'TeamB', 'TeamC'],
            'games': [30, 25, 35],
            'mpg': [25, 22, 28],
            'ppg': [12, 10, 15],
            'apg': [4, 3, 6],
            'rpg': [5, 4, 7],
            'spg': [1.2, 1.0, 1.5],
            'bpg': [0.6, 0.5, 0.9],
            'efficiency': [10, 8, 12],
            'true_shooting_percentage': [0.55, 0.52, 0.58],
            'three_pt_pct': [0.35, 0.32, 0.38]
        })

        # Test without ML predictions
        analyzer = ScoutingAnalyzer()
        prospects = analyzer.identify_scouting_targets(
            test_player_df, test_nba_df, test_intl_df, top_n=3
        )

        assert 'ml_multiplier' in prospects.columns, "Missing ml_multiplier"
        assert (prospects['ml_multiplier'] == 1.0).all(), "ml_multiplier should be 1.0 when no ML"
        print(f"\n✓ ml_multiplier = 1.0 when ML not available (documented)")

        # Test with ML predictions
        test_ml = pd.DataFrame({
            'player_id': ['p1', 'p2', 'p3'],
            'nba_success_prob': [0.7, 0.6, 0.8]
        })

        prospects_ml = analyzer.identify_scouting_targets(
            test_player_df, test_nba_df, test_intl_df, ml_predictions=test_ml, top_n=3
        )

        assert (prospects_ml['ml_multiplier'] != 1.0).any(), "ml_multiplier should vary with predictions"
        print(f"✓ ml_multiplier varies when ML available: {prospects_ml['ml_multiplier'].min():.2f}–{prospects_ml['ml_multiplier'].max():.2f}")

        print("\n✓ SCOUTING MODULE (FIXED): PASSED")

    except Exception as e:
        print(f"\n✗ ERROR: {e}")
        import traceback
        traceback.print_exc()


Overwriting modular/scouting/targets.py


In [9]:
%%writefile modular/reporting/generator.py
"""
Reporting and visualization module for basketball scouting.
Generates EDA plots, ML diagnostics, and scouting reports.
"""

import json
import pandas as pd
import numpy as np
from typing import Dict, Optional, List
from pathlib import Path

import matplotlib
matplotlib.use('Agg')  # Non-interactive backend
import matplotlib.pyplot as plt
import seaborn as sns
PLOTTING_AVAILABLE = True

from modular.config import Config
from modular.utils import format_stat, format_probability_range


class ReportGenerator:
    """Generates comprehensive scouting reports and visualizations."""

    @staticmethod
    def generate_eda_visualizations(player_df: pd.DataFrame,
                                    nba_df: pd.DataFrame,
                                    intl_df: pd.DataFrame,
                                    output_path: str = None):
        """
        Generate  EDA visualizations with data quality checks.

        Args:
            player_df: Player demographics
            nba_df: NBA statistics
            intl_df: International statistics
            output_path: Path to save figure (default from Config)
        """
        if not PLOTTING_AVAILABLE:
            print("\nMatplotlib not available; skipping EDA visualizations")
            return

        output_path = output_path or Config.EDA_VISUALIZATION

        print("\nGenerating EDA visualizations...")

        # Filter to 2021 season
        nba_2021 = nba_df[nba_df['season'] == 2021]
        intl_2021 = intl_df[intl_df['season'] == 2021]

        # Create figure with 3x3 grid
        fig = plt.figure(figsize=(20, 12))
        gs = fig.add_gridspec(3, 3, hspace=0.3, wspace=0.3)

        # Row 1: Performance distributions
        ax1 = fig.add_subplot(gs[0, 0])
        ax2 = fig.add_subplot(gs[0, 1])
        ax3 = fig.add_subplot(gs[0, 2])

        # Plot 1: True Shooting %
        if 'true_shooting_percentage' in nba_2021.columns:
            nba_ts = nba_2021['true_shooting_percentage'].dropna()
            intl_ts = intl_2021['true_shooting_percentage'].dropna()
            ax1.hist(nba_ts, bins=20, alpha=0.6, label=f'NBA (n={len(nba_ts)})', edgecolor='black')
            ax1.hist(intl_ts, bins=20, alpha=0.6, label=f'International (n={len(intl_ts)})', edgecolor='black')
            ax1.set_xlabel('True Shooting %')
            ax1.set_ylabel('Frequency')
            ax1.set_title('Shooting Efficiency (2021)')
            ax1.legend()
            ax1.grid(alpha=0.3)

        # Plot 2: Age vs MPG
        intl_with_age = intl_2021.merge(player_df[['player_id', 'age_2021']], on='player_id', how='left')
        if 'league' in intl_with_age.columns:
            leagues = intl_with_age['league'].value_counts().head(4).index
            for league in leagues:
                league_data = intl_with_age[intl_with_age['league'] == league]
                if len(league_data) > 0:
                    ax2.scatter(
                        league_data['age_2021'],
                        league_data['mpg'],
                        alpha=0.6, s=50, label=league, edgecolors='black', linewidth=0.5
                    )
            ax2.set_xlabel('Age (2021)')
            ax2.set_ylabel('Minutes Per Game')
            ax2.set_title('Age vs MPG (International 2021)')
            ax2.legend(loc='best', fontsize=8)
            ax2.grid(alpha=0.3)

        # Plot 3: 3PA Rate
        if 'three_point_attempt_rate' in nba_2021.columns:
            nba_3par = nba_2021['three_point_attempt_rate'].dropna()
            intl_3par = intl_2021['three_point_attempt_rate'].dropna()
            ax3.hist(nba_3par, bins=20, alpha=0.6, label=f'NBA (n={len(nba_3par)})', edgecolor='black')
            ax3.hist(intl_3par, bins=20, alpha=0.6, label=f'International (n={len(intl_3par)})', edgecolor='black')
            ax3.set_xlabel('3PA Rate')
            ax3.set_ylabel('Frequency')
            ax3.set_title('3PA Rate (2021)')
            ax3.legend()
            ax3.grid(alpha=0.3)

        # Row 2: Data quality
        ax4 = fig.add_subplot(gs[1, 0])
        ax5 = fig.add_subplot(gs[1, 1])
        ax6 = fig.add_subplot(gs[1, 2])

        # Plot 4-6: Data quality visualizations
        key_cols = ['games', 'minutes', 'points', 'assists', 'true_shooting_percentage']

        nba_missing = nba_df[key_cols].isnull().sum()
        ax4.barh(range(len(nba_missing)), nba_missing.values, color='steelblue', edgecolor='black')
        ax4.set_yticks(range(len(nba_missing)))
        ax4.set_yticklabels(nba_missing.index, fontsize=8)
        ax4.set_xlabel('Missing Values')
        ax4.set_title('NBA Data: Missing Values')
        ax4.grid(axis='x', alpha=0.3)

        intl_missing = intl_df[key_cols].isnull().sum()
        ax5.barh(range(len(intl_missing)), intl_missing.values, color='coral', edgecolor='black')
        ax5.set_yticks(range(len(intl_missing)))
        ax5.set_yticklabels(intl_missing.index, fontsize=8)
        ax5.set_xlabel('Missing Values')
        ax5.set_title('International Data: Missing Values')
        ax5.grid(axis='x', alpha=0.3)

        # Data completeness by season
        all_seasons = sorted(set(nba_df['season'].unique()) | set(intl_df['season'].unique()))
        nba_completeness = []
        intl_completeness = []

        for season in all_seasons:
            nba_season = nba_df[nba_df['season'] == season]
            intl_season = intl_df[intl_df['season'] == season]

            if len(nba_season) > 0:
                nba_complete = (1 - nba_season[key_cols].isnull().mean().mean()) * 100
                nba_completeness.append(nba_complete)
            else:
                nba_completeness.append(0)

            if len(intl_season) > 0:
                intl_complete = (1 - intl_season[key_cols].isnull().mean().mean()) * 100
                intl_completeness.append(intl_complete)
            else:
                intl_completeness.append(0)

        ax6.plot(all_seasons, nba_completeness, marker='o', label='NBA', linewidth=2)
        ax6.plot(all_seasons, intl_completeness, marker='s', label='International', linewidth=2)
        ax6.set_xlabel('Season')
        ax6.set_ylabel('Data Completeness (%)')
        ax6.set_title('Data Completeness by Season')
        ax6.legend()
        ax6.grid(alpha=0.3)
        ax6.set_ylim([0, 105])

        # Row 3: Additional analysis
        ax7 = fig.add_subplot(gs[2, 0])
        ax8 = fig.add_subplot(gs[2, 1])
        ax9 = fig.add_subplot(gs[2, 2])

        # Plot 7: FG% distribution
        for df, label, color in [(nba_df, 'NBA', 'steelblue'), (intl_df, 'International', 'coral')]:
            if 'two_points_made' in df.columns and 'two_points_attempted' in df.columns:
                valid_attempts = df['two_points_attempted'] > 10
                fg_pct = df.loc[valid_attempts, 'two_points_made'] / df.loc[valid_attempts, 'two_points_attempted']
                ax7.hist(fg_pct, bins=20, alpha=0.5, label=label, edgecolor='black', color=color)
        ax7.set_xlabel('2P%')
        ax7.set_ylabel('Frequency')
        ax7.set_title('Two-Point % Distribution\n(min 10 attempts)')
        ax7.legend()
        ax7.grid(alpha=0.3)

        # Plot 8: MPG boxplots
        nba_mpg = nba_df['mpg'].dropna()
        intl_mpg = intl_df['mpg'].dropna()
        ax8.boxplot([nba_mpg, intl_mpg], labels=['NBA', 'International'], widths=0.6)
        ax8.set_ylabel('Minutes Per Game')
        ax8.set_title('MPG Distribution by League')
        ax8.grid(axis='y', alpha=0.3)

        # Plot 9: Career length
        nba_career = nba_df.groupby('player_id')['season'].nunique()
        intl_career = intl_df.groupby('player_id')['season'].nunique()

        max_seasons = max(nba_career.max(), intl_career.max())
        ax9.hist(nba_career, bins=range(1, max_seasons+2), alpha=0.6, 
                label=f'NBA (n={len(nba_career)})', edgecolor='black', color='steelblue')
        ax9.hist(intl_career, bins=range(1, max_seasons+2), alpha=0.6,
                label=f'International (n={len(intl_career)})', edgecolor='black', color='coral')
        ax9.set_xlabel('Number of Seasons')
        ax9.set_ylabel('Number of Players')
        ax9.set_title('Career Length Distribution')
        ax9.legend()
        ax9.grid(alpha=0.3)

        plt.suptitle(' EDA with Data Quality Checks', fontsize=16, y=0.995)
        plt.savefig(output_path, dpi=150, bbox_inches='tight')
        plt.close()

        print(f"✓ Saved: {output_path}")

    @staticmethod
    def generate_ml_diagnostics(ml_artifacts: Dict, output_path: str = None):
        """
        Generate ML diagnostic plots (calibration, ROC, PR curves).

        Args:
            ml_artifacts: ML artifacts dictionary
            output_path: Path to save figure (default from Config)
        """
        if not PLOTTING_AVAILABLE:
            print("\nMatplotlib not available; skipping ML diagnostics")
            return

        if not ml_artifacts.get('enabled'):
            print("\nML model not available; skipping diagnostics")
            return

        output_path = output_path or Config.ML_DIAGNOSTIC_PLOTS

        print("\nGenerating ML diagnostic plots...")

        # Note: This is a simplified version - in full implementation,
        # you would store test predictions in ml_artifacts and plot them here
        print("✓ ML diagnostic plots (placeholder - implement with test set predictions)")

    @staticmethod
    def print_scouting_report(prospects: pd.DataFrame, ml_artifacts: Optional[Dict] = None):
        """
        Print formatted scouting report to console.

        Args:
            prospects: Top prospects DataFrame
            ml_artifacts: Optional ML metrics
        """
        print("\n" + "=" * 100)
        print("SCOUTING RECOMMENDATIONS")
        print("=" * 100)

        print("\nNote: Player identities are anonymized per assignment.")

        print("\n" + "-" * 100)
        print(f"TOP {Config.TOP_N_REPORT} SCOUTING TARGETS")
        print("-" * 100 + "\n")

        top_n = prospects.head(Config.TOP_N_REPORT)

        for idx, (_, player) in enumerate(top_n.iterrows(), 1):
            print(f"{'-' * 100}")
            print(f"RANK #{idx} | {player['first_name'].upper()} {player['last_name'].upper()}")
            print(f"{'-' * 100}")

            nba_status = "NBA experience" if player.get('has_nba_exp', False) else "No NBA experience"
            age_str = format_stat(player.get('age_2021'), 'int')
            league_str = format_stat(player.get('league'), 'str')
            team_str = format_stat(player.get('team'), 'str')

            print(f"Age {age_str} | {league_str} | {team_str} | {nba_status}")

            print("\n2021 Season:")
            print(f"  Games: {format_stat(player.get('games'), 'int')} | MPG: {format_stat(player.get('mpg'), 'float1')}")
            print(f"  PPG: {format_stat(player.get('ppg'), 'float1')} | APG: {format_stat(player.get('apg'), 'float1')} | RPG: {format_stat(player.get('rpg'), 'float1')}")
            print(f"  3P%: {format_stat(player.get('three_pt_pct'), 'pct')} | FT%: {format_stat(player.get('ft_pct'), 'pct')}")
            print(f"  EFF: {format_stat(player.get('efficiency'), 'float1')} | TS%: {format_stat(player.get('true_shooting_percentage'), 'pct')}")

            if pd.notna(player.get('ppg_change')) and player['ppg_change'] != 0:
                arrow = "↑" if player['ppg_change'] > 0 else "↓"
                print(f"\nTrend: {arrow} {abs(player['ppg_change']):.1f} PPG over career")

            if pd.notna(player.get('nba_success_prob')):
                prob_display = format_probability_range(player['nba_success_prob'])
                print(f"ML NBA success probability: {prob_display}")

            print(f"\nScout score: {player['scout_score']:.1f}\n")

        # Summary
        print("=" * 100)
        print("SUMMARY")
        print("=" * 100)

        if ml_artifacts and ml_artifacts.get('enabled'):
            m = ml_artifacts['metrics']
            print(f"Model: Calibrated Gradient Boosting")
            print(f"ROC-AUC (test): {m['test_auc']:.3f} | PR-AUC (test): {m['test_pr_auc']:.3f}")
            print(f"Brier Score (cal): {m['brier_score_calibrated']:.3f}")
            if 'precision_at_10' in m:
                print(f"Precision@10: {m['precision_at_10']:.1%}")

        print(f"\nTop {len(prospects)} prospects identified")
        print(f"With NBA experience: {int(prospects['has_nba_exp'].sum())}")
        print(f"International only: {int((~prospects['has_nba_exp']).sum())}")

    @staticmethod
    def save_outputs(prospects: pd.DataFrame, ml_artifacts: Dict, 
                    quality_issues: List[str], db_path: str):
        """
        Save all output files.

        Args:
            prospects: Top prospects DataFrame
            ml_artifacts: ML artifacts
            quality_issues: Data quality issues list
            db_path: Database path
        """
        print("\n" + "=" * 100)
        print("SAVING OUTPUTS")
        print("=" * 100)

        # Save scouting report CSV
        output_cols = [
            'first_name', 'last_name', 'age_2021', 'league', 'team',
            'games', 'mpg', 'ppg', 'apg', 'rpg', 'spg', 'bpg',
            'efficiency', 'true_shooting_percentage', 'fg_pct', 'three_pt_pct', 'ft_pct',
            'has_nba_exp', 'scout_score'
        ]

        if 'ppg_change' in prospects.columns:
            output_cols.append('ppg_change')
        if 'nba_success_prob' in prospects.columns:
            output_cols.append('nba_success_prob')

        available_cols = [col for col in output_cols if col in prospects.columns]
        prospects[available_cols].to_csv(Config.SCOUTING_REPORT_CSV, index=False)
        print(f"✓ Saved: {Config.SCOUTING_REPORT_CSV}")

        # Save ML metrics
        if ml_artifacts.get('enabled'):
            with open(Config.ML_METRICS_JSON, 'w') as f:
                json.dump(ml_artifacts['metrics'], f, indent=2)
            print(f"✓ Saved: {Config.ML_METRICS_JSON}")

        # Save data quality report
        with open(Config.DATA_QUALITY_REPORT, 'w') as f:
            f.write("DATA QUALITY REPORT\n")
            f.write("=" * 100 + "\n\n")
            f.write(f"Total issues identified: {len(quality_issues)}\n\n")
            for i, issue in enumerate(quality_issues, 1):
                f.write(f"{i}. {issue}\n")
        print(f"✓ Saved: {Config.DATA_QUALITY_REPORT}")

        print(f"\nDatabase: {db_path}")
        print("\nAll outputs saved successfully.")


def generate_comprehensive_report(prospects: pd.DataFrame, player_df: pd.DataFrame,
                                 nba_df: pd.DataFrame, intl_df: pd.DataFrame,
                                 ml_artifacts: Optional[Dict] = None,
                                 quality_issues: Optional[List[str]] = None,
                                 db_path: Optional[str] = None):
    """
    Main function to generate complete reporting output.

    Args:
        prospects: Top prospects DataFrame
        player_df: Player demographics
        nba_df: NBA statistics
        intl_df: International statistics
        ml_artifacts: Optional ML artifacts
        quality_issues: Optional quality issues list
        db_path: Optional database path
    """
    generator = ReportGenerator()

    # Generate EDA visualizations
    generator.generate_eda_visualizations(player_df, nba_df, intl_df)

    # Generate ML diagnostics
    if ml_artifacts:
        generator.generate_ml_diagnostics(ml_artifacts)

    # Print scouting report
    generator.print_scouting_report(prospects, ml_artifacts)

    # Save outputs
    if quality_issues and db_path:
        generator.save_outputs(prospects, ml_artifacts or {}, quality_issues, db_path)

        
        

Overwriting modular/reporting/generator.py


In [10]:
%%writefile modular/main.py
"""
main.py - FULLY CORRECTED VERSION

Main pipeline orchestration for basketball scouting analysis.
Coordinates all modules in the modular structure.

FIXES APPLIED:
1. Added ml_predictions storage to __init__
2. Fixed run_ml_pipeline() to extract predictions properly
3. Fixed run_scouting_pipeline() argument order and type
"""

# --- path bootstrap: works for "python modular/main.py" AND notebooks ---
import sys
from pathlib import Path

if __package__ in (None, ""):
    try:
        # When running as a file (script/module), __file__ exists
        PROJECT_ROOT = Path(__file__).resolve().parents[1]
    except NameError:
        # When pasted/executed inside a notebook/REPL, __file__ is missing
        PROJECT_ROOT = Path.cwd().resolve()
        # If CWD is ".../modular", go up one level to project root
        if PROJECT_ROOT.name == "modular":
            PROJECT_ROOT = PROJECT_ROOT.parent
    if str(PROJECT_ROOT) not in sys.path:
        sys.path.insert(0, str(PROJECT_ROOT))
# --- end bootstrap ---


from modular.config import Config
from modular.data.loader import DataLoader
from modular.database.manager import DatabaseManager
from modular.features.statistics import calculate_statistics
from modular.analysis.performance import analyze_performance_patterns
from modular.models.predictor import train_nba_success_model, NBASuccessPredictor
from modular.scouting.targets import identify_scouting_targets
from modular.reporting.generator import generate_comprehensive_report


class ScoutingPipeline:
    """Orchestrates the complete scouting analysis pipeline."""

    def __init__(self, data_dir: str = None, db_path: str = None):
        """
        Initialize the pipeline.

        Args:
            data_dir: Directory containing data files
            db_path: Path to database file
        """
        self.data_dir = data_dir or Config.DATA_DIR
        self.db_path = db_path or Config.DB_PATH

        # Components
        self.loader = DataLoader(self.data_dir)
        self.db = DatabaseManager(self.db_path)

        # Data holders
        self.player_df = None
        self.nba_df = None
        self.intl_df = None
        self.quality_issues = []
        self.traj_df = None
        self.ml_artifacts = None
        self.ml_predictions = None  # ✓ FIXED: Added predictions storage
        self.top_prospects = None

    def run_data_pipeline(self):
        """Step 1: Load and validate data."""
        print("\n" + "="*100)
        print("STEP 1: DATA LOADING AND VALIDATION")
        print("="*100)

        self.player_df, self.nba_df, self.intl_df, self.quality_issues = \
            self.loader.load_and_validate()

        print(f"\n✓ Loaded {len(self.player_df)} players")
        print(f"✓ Loaded {len(self.nba_df)} NBA records")
        print(f"✓ Loaded {len(self.intl_df)} International records")
        print(f"✓ Identified {len(self.quality_issues)} data quality issues")

        return self

    def run_database_pipeline(self):
        """Step 2: Setup database and load data."""
        print("\n" + "="*100)
        print("STEP 2: DATABASE SETUP AND ETL")
        print("="*100)

        # DEBUG ISSUE #1: Track data drift before/after database
        print(f"\n[DEBUG ISSUE #1] Pre-database counts:")
        print(f"  player_df: {len(self.player_df)} rows")
        print(f"  nba_df: {len(self.nba_df)} rows")
        print(f"  intl_df: {len(self.intl_df)} rows")

        self.db.connect()
        self.db.create_schema()
        self.db.create_indexes()

        # FIX ISSUE #1: Capture filtered DataFrames
        filtered_nba, filtered_intl = self.db.load_data(
            self.player_df,
            self.nba_df,
            self.intl_df,
            self.quality_issues
        )

        # DEBUG ISSUE #1: Check what was filtered
        nba_removed = len(self.nba_df) - len(filtered_nba)
        intl_removed = len(self.intl_df) - len(filtered_intl)
        print(f"\n[DEBUG ISSUE #1] Database filtering results:")
        print(f"  nba_df: {len(self.nba_df)} → {len(filtered_nba)} ({nba_removed} removed)")
        print(f"  intl_df: {len(self.intl_df)} → {len(filtered_intl)} ({intl_removed} removed)")

        # FIX ISSUE #1: Update pipeline DataFrames with filtered versions
        self.nba_df = filtered_nba
        self.intl_df = filtered_intl
        print(f"[DEBUG ISSUE #1] ✓ Pipeline DataFrames updated to filtered versions")

        self.db.run_sanity_checks()

        print(f"\n✓ Database created: {self.db_path}")

        return self

    def run_statistics_pipeline(self):
        """Step 3: Calculate statistics."""
        print("\n" + "="*100)
        print("STEP 3: STATISTICS CALCULATION")
        print("="*100)

        self.nba_df, self.intl_df = calculate_statistics(
            self.nba_df, 
            self.intl_df
        )

        # Display some calculated stats
        calc_cols = [col for col in Config.SCHEMAS.CALCULATED_ALL 
                     if col in self.intl_df.columns]
        print(f"\n✓ Calculated {len(calc_cols)} statistics")
        print(f"  Per-game: {', '.join(Config.SCHEMAS.CALCULATED_PER_GAME[:5])}")
        print(f"  Shooting: {', '.join(Config.SCHEMAS.CALCULATED_SHOOTING)}")
        print(f"  Efficiency: {', '.join(Config.SCHEMAS.CALCULATED_EFFICIENCY)}")

        return self

    def run_analysis_pipeline(self):
        """Step 4: Analyze performance patterns."""
        print("\n" + "="*100)
        print("STEP 4: PERFORMANCE ANALYSIS")
        print("="*100)

        self.traj_df = analyze_performance_patterns(
            self.intl_df,
            self.nba_df
        )

        if self.traj_df is not None and len(self.traj_df) > 0:
            print(f"\n✓ Analyzed {len(self.traj_df)} player trajectories")
            print(f"  Average PPG change: {self.traj_df['ppg_change'].mean():+.2f}")
            print(f"  Average efficiency change: {self.traj_df['efficiency_change'].mean():+.2f}")

        return self

    def run_ml_pipeline(self):
        """Step 5: Train ML model and extract predictions."""
        print("\n" + "="*100)
        print("STEP 5: MACHINE LEARNING MODEL")
        print("="*100)

        try:
            # Train model - returns ml_artifacts dict
            self.ml_artifacts = train_nba_success_model(
                self.player_df, self.nba_df, self.intl_df
            )

            if self.ml_artifacts['enabled']:
                metrics = self.ml_artifacts['metrics']
                print(f"\n✓ ML model trained successfully")
                print(f"  ROC-AUC: {metrics['test_auc']:.3f}")
                print(f"  PR-AUC: {metrics['test_pr_auc']:.3f}")
                print(f"  Brier Score: {metrics['brier_score_calibrated']:.3f}")

                # ✓ FIXED: Extract predictions from model artifacts
                # The predictor stores predictions in a separate location
                # We need to extract them or generate them fresh
                print("\n  Generating predictions for current prospects...")

                # Create predictor instance to access predictions
                predictor = NBASuccessPredictor()
                predictor.ml_artifacts = self.ml_artifacts

                # Generate predictions for current season
                predictions_df = predictor.predict_current_prospects(
                    self.intl_df, 
                    season=Config.CURRENT_SEASON
                )

                # Store predictions separately for scouting module
                self.ml_predictions = predictions_df
                print(f"  ✓ Generated {len(predictions_df)} predictions")

                # Debug output (optional - remove in production)
                print(f"\n  [DEBUG] Predictions DataFrame:")
                print(f"    Type: {type(self.ml_predictions)}")
                print(f"    Shape: {self.ml_predictions.shape}")
                print(f"    Columns: {list(self.ml_predictions.columns)}")
            else:
                self.ml_predictions = None

        except Exception as e:
            print(f"\n⚠ ML training error: {e}")
            import traceback
            traceback.print_exc()
            self.ml_artifacts = {'enabled': False}
            self.ml_predictions = None

        return self

    def run_scouting_pipeline(self):
        """Step 6: Identify scouting targets."""
        print("\n" + "="*100)
        print("STEP 6: SCOUTING TARGET IDENTIFICATION")
        print("="*100)

        try:
            # Debug output (optional - remove in production)
            print("\n[DEBUG] Inputs to scouting function:")
            print(f"  player_df: {type(self.player_df)} ({len(self.player_df)} rows)")
            print(f"  nba_df: {type(self.nba_df)} ({len(self.nba_df)} rows)")
            print(f"  intl_df: {type(self.intl_df)} ({len(self.intl_df)} rows)")
            if self.traj_df is not None:
                print(f"  traj_df: {type(self.traj_df)} ({len(self.traj_df)} rows)")
            if self.ml_predictions is not None:
                print(f"  ml_predictions: {type(self.ml_predictions)} ({len(self.ml_predictions)} rows)")

            # ✓ FIXED: Correct argument order and pass predictions DataFrame
            # Using named arguments to prevent order errors
            self.top_prospects = identify_scouting_targets(
                player_df=self.player_df,              # ✓ Correct
                nba_df=self.nba_df,                    # ✓ Correct order
                intl_df=self.intl_df,                  # ✓ Correct order
                traj_df=self.traj_df,                  # ✓ Correct
                ml_predictions=self.ml_predictions,    # ✓ DataFrame, not dict!
                season=Config.CURRENT_SEASON,
                top_n=Config.TOP_N_PROSPECTS
            )

            if self.top_prospects is not None and len(self.top_prospects) > 0:
                print(f"\n✓ Identified {len(self.top_prospects)} top prospects")
                print(f"  Top scout score: {self.top_prospects.iloc[0]['scout_score']:.1f}")
                print(f"  Average age: {self.top_prospects['age_2021'].mean():.1f} years")

        except Exception as e:
            print(f"\n⚠ Scouting error: {e}")
            import traceback
            traceback.print_exc()

        return self

    def run_reporting_pipeline(self):
        """Step 7: Generate reports."""
        print("\n" + "="*100)
        print("STEP 7: REPORT GENERATION")
        print("="*100)

        try:
            if self.top_prospects is not None and len(self.top_prospects) > 0:
                generate_comprehensive_report(
                    self.top_prospects, self.player_df, 
                    self.nba_df, self.intl_df,
                    self.ml_artifacts, self.quality_issues, self.db_path
                )

                print(f"\n✓ Generated scouting report")
                print(f"✓ Saved outputs to current directory")
            else:
                print("\n⚠ No prospects to report")

        except Exception as e:
            print(f"\n⚠ Reporting error: {e}")
            import traceback
            traceback.print_exc()

        return self

    def run_full_pipeline(self):
        """Execute the complete analysis pipeline."""
        print("\n" + "="*100)
        print("SACRAMENTO KINGS - INTERNATIONAL SCOUTING ANALYSIS")
        print("MODULAR PIPELINE v2.1 (FIXED)")
        print("="*100)

        print(f"\nConfiguration:")
        print(f"  Data Directory: {self.data_dir}")
        print(f"  Database: {self.db_path}")
        print(f"  Current Season: {Config.CURRENT_SEASON}")
        print(f"  Min Games: {Config.MIN_GAMES_THRESHOLD}")
        print(f"  Min MPG: {Config.MIN_MPG_THRESHOLD}")
        print(f"  Max Age: {Config.MAX_AGE_THRESHOLD}")

        try:
            self.run_data_pipeline()
            self.run_database_pipeline()
            self.run_statistics_pipeline()
            self.run_analysis_pipeline()
            self.run_ml_pipeline()
            self.run_scouting_pipeline()
            self.run_reporting_pipeline()

            print("\n" + "="*100)
            print("PIPELINE COMPLETE ✓")
            print("="*100)

            self._print_summary()

        except Exception as e:
            print(f"\n✗ Pipeline failed: {e}")
            import traceback
            traceback.print_exc()

        finally:
            if self.db.conn is not None:
                self.db.close()

    def _print_summary(self):
        """Print pipeline execution summary."""
        print("\nPipeline Summary:")
        print("─" * 100)

        print(f"\n📊 Data Processing:")
        print(f"   Players: {len(self.player_df):,}")
        print(f"   NBA Records: {len(self.nba_df):,}")
        print(f"   International Records: {len(self.intl_df):,}")
        print(f"   Quality Issues: {len(self.quality_issues)}")

        if self.traj_df is not None and len(self.traj_df) > 0:
            print(f"\n📈 Performance Analysis:")
            print(f"   Players with Trajectories: {len(self.traj_df)}")
            improving = (self.traj_df['ppg_change'] > 3).sum()
            print(f"   Strongly Improving: {improving}")

        if self.ml_artifacts and self.ml_artifacts.get('enabled'):
            print(f"\n🤖 Machine Learning:")
            metrics = self.ml_artifacts['metrics']
            print(f"   ROC-AUC: {metrics['test_auc']:.3f}")
            print(f"   PR-AUC: {metrics['test_pr_auc']:.3f}")

        if self.top_prospects is not None and len(self.top_prospects) > 0:
            print(f"\n🎯 Scouting:")
            print(f"   Top Prospects: {len(self.top_prospects)}")
            print(f"   With NBA Experience: {int(self.top_prospects['has_nba_exp'].sum())}")

        print("\n" + "─" * 100)

    def cleanup(self):
        """Cleanup resources."""
        if self.db.conn is not None:
            self.db.close()


def main():
    """Main entry point for the scouting analysis."""
    pipeline = ScoutingPipeline()
    pipeline.run_full_pipeline()


if __name__ == "__main__":
    # Smoke test before running full pipeline
    print("Running pre-flight checks...")

    # Check that Config is accessible
    assert Config.CURRENT_SEASON == 2021, "Config not loaded correctly"
    print("✓ Config loaded")

    # Check data directory exists
    data_dir = Path(Config.DATA_DIR)
    if not data_dir.exists():
        print(f"⚠ Warning: Data directory not found: {Config.DATA_DIR}")
        print("  Please ensure data files are in the correct location")
    else:
        print(f"✓ Data directory found: {Config.DATA_DIR}")

    # Run main pipeline
    main()


Overwriting modular/main.py


# Tests

In [11]:
%%writefile modular/run_tests.py
#!/usr/bin/env python3
"""
Integration tests for the modular basketball scouting pipeline.
Tests module interactions and data flow.
"""

import sys
import traceback
from pathlib import Path

def test_module_imports():
    """Test that all modules can be imported."""
    print("=" * 60)
    print("TEST 1: MODULE IMPORTS")
    print("=" * 60)
    
    try:
        from data import DataLoader
        print("✓ DataLoader imported")
        
        from database import DatabaseManager
        print("✓ DatabaseManager imported")
        
        from features import calculate_statistics, StatisticsCalculator
        print("✓ Statistics functions imported")
        
        from analysis import analyze_performance_patterns, PerformanceAnalyzer
        print("✓ Performance analysis imported")
        
        from models import train_nba_success_model, NBASuccessPredictor
        print("✓ ML models imported")
        
        from scouting import identify_scouting_targets, ScoutingAnalyzer
        print("✓ Scouting analysis imported")
        
        from reporting import generate_comprehensive_report, ReportGenerator
        print("✓ Reporting functions imported")
        
        print("\n✓ All module imports successful!")
        return True
        
    except ImportError as e:
        print(f"\n✗ Import error: {e}")
        return False
    except Exception as e:
        print(f"\n✗ Error: {e}")
        return False

def test_utils_functions():
    """Test utility functions work correctly."""
    print("\n" + "=" * 60)
    print("TEST 2: UTILITY FUNCTIONS")
    print("=" * 60)
    
    try:
        from utils import safe_div, create_player_id, calculate_age_bonus
        
        # Test safe_div
        assert safe_div(10, 2) == 5.0, "safe_div failed"
        print("✓ safe_div works")
        
        # Test create_player_id
        import pandas as pd
        pid = create_player_id(pd.Series(['John']), pd.Series(['Doe']))
        assert pid.iloc[0] == 'john_doe', "create_player_id failed"
        print("✓ create_player_id works")
        
        # Test age bonus
        assert calculate_age_bonus(22) == 1.3, "age bonus failed"
        print("✓ calculate_age_bonus works")
        
        print("\n✓ All utility functions work!")
        return True
        
    except Exception as e:
        print(f"\n✗ Error: {e}")
        traceback.print_exc()
        return False

def test_data_loader():
    """Test data loader with synthetic data."""
    print("\n" + "=" * 60)
    print("TEST 3: DATA LOADER")
    print("=" * 60)
    
    try:
        from data import DataLoader
        import tempfile
        import json
        import shutil
        
        # Create temp directory with test data
        temp_dir = tempfile.mkdtemp()
        
        test_data = [
            {"first_name": "Test", "last_name": "Player", "birth_date": "1995-01-15"}
        ]
        
        with open(Path(temp_dir) / 'player.json', 'w') as f:
            json.dump(test_data, f)
        with open(Path(temp_dir) / 'nba_box_player_season.json', 'w') as f:
            json.dump([], f)
        with open(Path(temp_dir) / 'international_box_player_season.json', 'w') as f:
            json.dump([], f)
        
        # Test loader
        loader = DataLoader(data_dir=temp_dir)
        player_df, nba_df, intl_df, issues = loader.load_and_validate()
        
        assert len(player_df) == 1, "Wrong player count"
        assert 'player_id' in player_df.columns, "Missing player_id"
        print("✓ DataLoader works with synthetic data")
        
        # Cleanup
        shutil.rmtree(temp_dir)
        print("✓ Cleanup successful")
        
        return True
        
    except Exception as e:
        print(f"\n✗ Error: {e}")
        traceback.print_exc()
        return False

def test_database_manager():
    """Test database manager with in-memory database."""
    print("\n" + "=" * 60)
    print("TEST 4: DATABASE MANAGER")
    print("=" * 60)
    
    try:
        from database import DatabaseManager
        import pandas as pd
        
        # Create test data
        player_df = pd.DataFrame({
            'player_id': ['test_player'],
            'first_name': ['Test'],
            'last_name': ['Player'],
            'birth_date': ['1995-01-15'],
            'birth_year': [1995],
            'age_2021': [26]
        })
        
        nba_df = pd.DataFrame({
            'player_id': ['test_player'],
            'season': [2021],
            'team': ['TestTeam'],
            'games': [50],
            'minutes': [1200.0],
            'points': [600.0],
            'assists': [200.0],
            'offensive_rebounds': [50.0],
            'defensive_rebounds': [150.0]
        })
        
        intl_df = pd.DataFrame()
        quality_issues = []
        
        # Test database
        db = DatabaseManager(db_path=':memory:')
        db.connect()
        db.create_schema()
        db.create_indexes()
        db.load_data(player_df, nba_df, intl_df, quality_issues)
        
        # Verify data
        query = "SELECT COUNT(*) FROM players"
        count = pd.read_sql_query(query, db.conn).iloc[0, 0]
        assert count == 1, f"Expected 1 player, got {count}"
        
        db.close()
        print("✓ DatabaseManager works with in-memory database")
        
        return True
        
    except Exception as e:
        print(f"\n✗ Error: {e}")
        traceback.print_exc()
        return False

def test_statistics_calculator():
    """Test statistics calculator."""
    print("\n" + "=" * 60)
    print("TEST 5: STATISTICS CALCULATOR")
    print("=" * 60)
    
    try:
        from features import StatisticsCalculator
        import pandas as pd
        
        # Create test data
        test_data = pd.DataFrame({
            'games': [50, 60, 70],
            'minutes': [1200, 1500, 1800],
            'points': [600, 900, 1200],
            'assists': [200, 300, 250],
            'offensive_rebounds': [50, 40, 60],
            'defensive_rebounds': [150, 180, 200],
            'two_points_made': [150, 200, 250],
            'two_points_attempted': [300, 400, 500],
            'three_points_made': [50, 100, 100],
            'three_points_attempted': [150, 300, 300],
            'free_throws_made': [100, 100, 100],
            'free_throws_attempted': [120, 130, 140]
        })
        
        # Test calculator
        calc = StatisticsCalculator()
        result = calc.calculate_all_statistics(test_data, 'Test')
        
        assert 'ppg' in result.columns, "Missing PPG column"
        assert 'efficiency' in result.columns, "Missing efficiency column"
        assert result['ppg'].iloc[0] == 12.0, "PPG calculation wrong"
        
        print("✓ StatisticsCalculator works correctly")
        
        return True
        
    except Exception as e:
        print(f"\n✗ Error: {e}")
        traceback.print_exc()
        return False

def main():
    """Run all integration tests."""
    print("BASKETBALL SCOUTING PIPELINE - INTEGRATION TESTS")
    print("=" * 60)
    
    tests = [
        test_module_imports,
        test_utils_functions,
        test_data_loader,
        test_database_manager,
        test_statistics_calculator
    ]
    
    passed = 0
    total = len(tests)
    
    for test in tests:
        try:
            if test():
                passed += 1
        except Exception as e:
            print(f"\n✗ Test {test.__name__} failed with exception: {e}")
    
    print("\n" + "=" * 60)
    print("INTEGRATION TEST RESULTS")
    print("=" * 60)
    print(f"Passed: {passed}/{total}")
    
    if passed == total:
        print("✓ ALL TESTS PASSED!")
        print("✓ Pipeline is ready for use")
        return True
    else:
        print("✗ SOME TESTS FAILED")
        print("✗ Check errors above and fix issues")
        return False

if __name__ == "__main__":
    success = main()
    sys.exit(0 if success else 1)

Overwriting modular/run_tests.py


# Frontend

In [12]:
%%writefile app.py
"""
 Streamlit App for International Basketball Scouting
Comprehensive EDA, Model Diagnostics, and Recommendations

This version adds:
- Interactive Plotly visualizations
- Comprehensive statistical analysis
- Data quality insights
- Model performance breakdowns
- Assignment-specific documentation
"""

import json
import math
import pandas as pd
import numpy as np
import streamlit as st
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

st.set_page_config(page_title="Kings | Intl Scouting", layout="wide", initial_sidebar_state="expanded")

# =============================================================================
# DATA LOADING
# =============================================================================

@st.cache_data
def load_reports():
    """Load all report data with error handling."""
    df = pd.read_csv("final_scouting_report.csv")

    try:
        with open("ml_metrics.json", "r") as f:
            metrics = json.load(f)
    except Exception:
        metrics = {}

    try:
        fi = pd.read_csv("feature_importance.csv")
    except Exception:
        fi = pd.DataFrame()

    return df, metrics, fi


@st.cache_data
def calculate_statistics(df):
    """Calculate summary statistics for the dataset."""
    stats = {
        'total_prospects': len(df),
        'avg_age': df['age_2021'].mean(),
        'avg_ppg': df['ppg'].mean(),
        'avg_mpg': df['mpg'].mean(),
        'with_nba_exp': df['has_nba_exp'].sum(),
        'leagues': df['league'].nunique() if 'league' in df.columns else 0,
        'avg_scout_score': df['scout_score'].mean(),
        'improving_players': (df['ppg_change'] > 3).sum() if 'ppg_change' in df.columns else 0,
    }
    return stats


# =============================================================================
# VISUALIZATION FUNCTIONS
# =============================================================================

def create_age_distribution_plot(df):
    """Create age distribution histogram."""
    fig = px.histogram(
        df,
        x='age_2021',
        nbins=15,
        title='Age Distribution of Top Prospects',
        labels={'age_2021': 'Age (2021)', 'count': 'Number of Players'},
        color_discrete_sequence=['#5D3A9B']
    )
    fig.update_layout(
        showlegend=False,
        height=400,
        hovermode='x unified'
    )
    fig.add_vline(x=df['age_2021'].mean(), line_dash="dash",
                  annotation_text=f"Mean: {df['age_2021'].mean():.1f}",
                  line_color="red")
    return fig


def create_performance_scatter(df):
    """Create PPG vs Efficiency scatter plot."""
    fig = px.scatter(
        df,
        x='ppg',
        y='efficiency',
        size='mpg',
        color='age_2021',
        hover_data=['first_name', 'last_name', 'league', 'team'],
        title='Performance Analysis: PPG vs Efficiency',
        labels={
            'ppg': 'Points Per Game',
            'efficiency': 'Efficiency Rating',
            'age_2021': 'Age',
            'mpg': 'Minutes Per Game'
        },
        color_continuous_scale='Viridis'
    )
    fig.update_layout(height=500)
    return fig


def create_league_comparison(df):
    """Create league-wise performance comparison."""
    if 'league' not in df.columns:
        return None

    league_stats = df.groupby('league').agg({
        'ppg': 'mean',
        'apg': 'mean',
        'rpg': 'mean',
        'efficiency': 'mean',
        'scout_score': 'mean',
        'first_name': 'count'
    }).reset_index()
    league_stats.columns = ['league', 'PPG', 'APG', 'RPG', 'Efficiency', 'Scout Score', 'Players']

    # Sort by number of players
    league_stats = league_stats.sort_values('Players', ascending=False)

    fig = make_subplots(
        rows=1, cols=2,
        subplot_titles=('Average Stats by League', 'Number of Prospects by League'),
        specs=[[{'type': 'bar'}, {'type': 'bar'}]]
    )

    # Stats comparison
    fig.add_trace(
        go.Bar(name='PPG', x=league_stats['league'], y=league_stats['PPG'], marker_color='#5D3A9B'),
        row=1, col=1
    )
    fig.add_trace(
        go.Bar(name='APG', x=league_stats['league'], y=league_stats['APG'], marker_color='#E8927C'),
        row=1, col=1
    )
    fig.add_trace(
        go.Bar(name='RPG', x=league_stats['league'], y=league_stats['RPG'], marker_color='#7FC8A9'),
        row=1, col=1
    )

    # Player count
    fig.add_trace(
        go.Bar(x=league_stats['league'], y=league_stats['Players'],
               marker_color='#5D3A9B', showlegend=False),
        row=1, col=2
    )

    fig.update_xaxes(title_text="League", row=1, col=1)
    fig.update_xaxes(title_text="League", row=1, col=2)
    fig.update_yaxes(title_text="Per Game Average", row=1, col=1)
    fig.update_yaxes(title_text="Number of Players", row=1, col=2)

    fig.update_layout(height=450, showlegend=True, hovermode='x unified')
    return fig


def create_shooting_analysis(df):
    """Create comprehensive shooting analysis."""
    fig = make_subplots(
        rows=1, cols=3,
        subplot_titles=('3-Point %', 'Free Throw %', 'True Shooting %'),
        specs=[[{'type': 'box'}, {'type': 'box'}, {'type': 'box'}]]
    )

    # 3PT%
    fig.add_trace(
        go.Box(y=df['three_pt_pct'], name='3P%', marker_color='#5D3A9B', showlegend=False),
        row=1, col=1
    )

    # FT%
    fig.add_trace(
        go.Box(y=df['ft_pct'], name='FT%', marker_color='#E8927C', showlegend=False),
        row=1, col=2
    )

    # TS%
    fig.add_trace(
        go.Box(y=df['true_shooting_percentage'], name='TS%', marker_color='#7FC8A9', showlegend=False),
        row=1, col=3
    )

    fig.update_yaxes(title_text="Percentage", row=1, col=1)
    fig.update_yaxes(title_text="Percentage", row=1, col=2)
    fig.update_yaxes(title_text="Percentage", row=1, col=3)

    fig.update_layout(height=400, title_text="Shooting Efficiency Analysis")
    return fig


def create_improvement_analysis(df):
    """Create player improvement trend analysis."""
    if 'ppg_change' not in df.columns:
        return None

    # Filter to players with meaningful data
    improving_df = df[df['ppg_change'].notna()].copy()
    improving_df['improvement_category'] = pd.cut(
        improving_df['ppg_change'],
        bins=[-float('inf'), -3, 0, 3, 5, float('inf')],
        labels=['Declining (< -3)', 'Slight Decline (0 to -3)',
                'Stable (0 to 3)', 'Improving (3-5)', 'Strong Growth (> 5)']
    )

    category_counts = improving_df['improvement_category'].value_counts()

    fig = go.Figure(data=[
        go.Bar(
            x=category_counts.index.astype(str),
            y=category_counts.values,
            marker_color=['#E74C3C', '#F39C12', '#95A5A6', '#2ECC71', '#27AE60'],
            text=category_counts.values,
            textposition='outside'
        )
    ])

    fig.update_layout(
        title='Player Development Trends (PPG Change)',
        xaxis_title='Improvement Category',
        yaxis_title='Number of Players',
        height=450,
        showlegend=False
    )

    return fig


def create_success_probability_distribution(df):
    """Create NBA success probability distribution."""
    if 'nba_success_prob' not in df.columns:
        return None

    prob_df = df[df['nba_success_prob'].notna()].copy()

    fig = px.histogram(
        prob_df,
        x='nba_success_prob',
        nbins=20,
        title='NBA Success Probability Distribution',
        labels={'nba_success_prob': 'Success Probability', 'count': 'Number of Players'},
        color_discrete_sequence=['#5D3A9B']
    )

    fig.add_vline(
        x=prob_df['nba_success_prob'].median(),
        line_dash="dash",
        annotation_text=f"Median: {prob_df['nba_success_prob'].median():.1%}",
        line_color="red"
    )

    fig.update_layout(height=400, showlegend=False)
    fig.update_xaxes(tickformat='.0%')

    return fig


def create_correlation_heatmap(df):
    """Create correlation heatmap for key statistics."""
    numeric_cols = ['ppg', 'apg', 'rpg', 'spg', 'bpg', 'efficiency',
                    'true_shooting_percentage', 'three_pt_pct', 'ft_pct',
                    'mpg', 'scout_score']

    # Filter to available columns
    available_cols = [col for col in numeric_cols if col in df.columns]
    corr_matrix = df[available_cols].corr()

    fig = go.Figure(data=go.Heatmap(
        z=corr_matrix.values,
        x=corr_matrix.columns,
        y=corr_matrix.columns,
        colorscale='RdBu_r',
        zmid=0,
        text=np.round(corr_matrix.values, 2),
        texttemplate='%{text}',
        textfont={"size": 8},
        colorbar=dict(title="Correlation")
    ))

    fig.update_layout(
        title='Correlation Matrix: Key Performance Indicators',
        height=600,
        width=800
    )

    return fig


def create_feature_importance_plot(fi_df):
    """Create feature importance visualization."""
    if fi_df.empty:
        return None

    # Sort and take top 15
    fi_sorted = fi_df.sort_values('importance_perm', ascending=False).head(15)

    fig = go.Figure()

    fig.add_trace(go.Bar(
        y=fi_sorted['feature'],
        x=fi_sorted['importance_perm'],
        orientation='h',
        marker_color='#5D3A9B',
        error_x=dict(
            type='data',
            array=fi_sorted['importance_perm_std'],
            visible=True
        ),
        text=np.round(fi_sorted['importance_perm'], 3),
        textposition='outside'
    ))

    fig.update_layout(
        title='Top 15 Features by Permutation Importance',
        xaxis_title='Importance (PR-AUC Drop)',
        yaxis_title='Feature',
        height=500,
        showlegend=False
    )

    return fig


def create_scout_score_breakdown(df):
    """Create scout score component analysis."""
    top_10 = df.nlargest(10, 'scout_score')[['first_name', 'last_name', 'scout_score',
                                               'age_2021', 'ppg', 'efficiency',
                                               'nba_success_prob']].copy()

    top_10['player_name'] = top_10['first_name'] + ' ' + top_10['last_name']

    fig = go.Figure()

    # Scout score bars
    fig.add_trace(go.Bar(
        x=top_10['player_name'],
        y=top_10['scout_score'],
        name='Scout Score',
        marker_color='#5D3A9B',
        text=np.round(top_10['scout_score'], 1),
        textposition='outside'
    ))

    fig.update_layout(
        title='Top 10 Prospects: Scout Score Rankings',
        xaxis_title='Player',
        yaxis_title='Scout Score',
        height=450,
        showlegend=False,
        xaxis_tickangle=-45
    )

    return fig


# =============================================================================
# MAIN APP
# =============================================================================

# Load data
df, metrics, fi = load_reports()
stats = calculate_statistics(df)

# =============================================================================
# SIDEBAR
# =============================================================================

st.sidebar.header("🏀 Filters")

# League filter
if 'league' in df.columns:
    leagues = sorted([l for l in df['league'].dropna().unique()])
    sel_leagues = st.sidebar.multiselect("League", leagues, default=leagues)
else:
    sel_leagues = []

# MPG filter
min_mpg = float(df["mpg"].min()) if "mpg" in df else 0.0
max_mpg = float(df["mpg"].max()) if "mpg" in df else 40.0
mpg_range = st.sidebar.slider("Minutes per game", min_mpg, max_mpg, (20.0, max_mpg))

# Age filter
age_max = st.sidebar.slider("Max age (2021)", 20, int(df["age_2021"].max()), 30)

# NBA experience filter
exp_opt = st.sidebar.selectbox("NBA experience", ["All", "With NBA exp", "Intl only"])

# Display options
top_k = st.sidebar.slider("Top-K to highlight", 5, 30, 20)
sort_by = st.sidebar.selectbox("Sort by", ["scout_score", "nba_success_prob", "ppg", "efficiency"])

st.sidebar.markdown("---")
st.sidebar.caption("📊 Anonymized data per assignment")
st.sidebar.caption("🎯 Probabilities are calibrated")

# =============================================================================
# FILTER DATA
# =============================================================================

f = df.copy()

if 'league' in f and sel_leagues:
    f = f[f["league"].isin(sel_leagues)]

f = f[(f["mpg"] >= mpg_range[0]) & (f["mpg"] <= mpg_range[1])]
f = f[f["age_2021"] <= age_max]

if exp_opt == "With NBA exp":
    f = f[f["has_nba_exp"] == True]
elif exp_opt == "Intl only":
    f = f[f["has_nba_exp"] == False]

# Sort and rank
if sort_by in f.columns:
    f = f.sort_values(sort_by, ascending=False)
f["rank"] = range(1, len(f) + 1)

# =============================================================================
# HEADER METRICS
# =============================================================================

st.title("🏀 Sacramento Kings: International Scouting Analysis")
st.markdown("**Data**: NBA + European Leagues (2010–2021) | **Purpose**: Assignment Demonstration")

col1, col2, col3, col4 = st.columns(4)

with col1:
    st.metric("📈 PR-AUC", f"{metrics.get('test_pr_auc', 0):.3f}" if metrics else "—",
              help="Precision-Recall AUC (primary metric for imbalanced data)")
with col2:
    st.metric("📊 ROC-AUC", f"{metrics.get('test_auc', 0):.3f}" if metrics else "—",
              help="Receiver Operating Characteristic AUC")
with col3:
    st.metric("🎯 Precision@10", f"{metrics.get('precision_at_10', 0):.1%}" if metrics else "—",
              help="Precision in top 10 predictions")
with col4:
    st.metric("📉 Brier Score", f"{metrics.get('brier_score_calibrated', 0):.3f}" if metrics else "—",
              help="Calibration quality (lower is better)")

st.markdown("---")

# =============================================================================
# TABS
# =============================================================================

tab_rec, tab_eda, tab_model, tab_process, tab_limits = st.tabs([
    "🎯 Recommendations",
    "📊 Exploratory Data Analysis",
    "🤖 Model Diagnostics",
    "📋 Process & Methodology",
    "⚠️ Limitations"
])

# =============================================================================
# TAB 1: RECOMMENDATIONS
# =============================================================================

with tab_rec:
    st.subheader("Top Scouting Targets")
    st.markdown(f"Showing **{len(f)}** prospects matching filters (sorted by {sort_by})")

    # Display table
    display_cols = [c for c in [
        "rank", "first_name", "last_name", "age_2021", "league", "team",
        "games", "mpg", "ppg", "apg", "rpg", "three_pt_pct",
        "true_shooting_percentage", "efficiency", "nba_success_prob", "scout_score"
    ] if c in f.columns]

    st.dataframe(
        f[display_cols].head(200).style.background_gradient(
            subset=['scout_score'], cmap='Greens'
        ),
        use_container_width=True,
        height=400
    )

    st.download_button(
        "📥 Download Filtered CSV",
        f.to_csv(index=False).encode("utf-8"),
        file_name="scouting_recs_filtered.csv",
        mime="text/csv"
    )

    st.markdown("---")

    # Player details
    st.subheader("🔍 Individual Player Analysis")

    if not f.empty:
        player_names = f["first_name"] + " " + f["last_name"]
        selected_player = st.selectbox("Select player for detailed view:", player_names)

        row = f[player_names == selected_player].iloc[0]

        col_a, col_b, col_c, col_d = st.columns(4)

        with col_a:
            st.metric("Minutes/Game", f"{row.get('mpg', 0):.1f}")
        with col_b:
            st.metric("Points/Game", f"{row.get('ppg', 0):.1f}")
        with col_c:
            st.metric("Efficiency", f"{row.get('efficiency', 0):.1f}")
        with col_d:
            st.metric("True Shooting %", f"{row.get('true_shooting_percentage', 0):.1%}")

        st.markdown(f"""
        **Profile**: {row.get('first_name', '')} {row.get('last_name', '')}
        **Team**: {row.get('team', '—')} | **League**: {row.get('league', '—')} | **Age**: {int(row.get('age_2021', 0))}
        **Scout Score**: {row.get('scout_score', 0):.1f}
        """)

        if 'nba_success_prob' in row and not math.isnan(row.get("nba_success_prob", float("nan"))):
            prob = row['nba_success_prob']
            st.info(f"🎯 **Calibrated NBA Success Probability**: {prob:.1%}")

        if 'ppg_change' in row and not math.isnan(row.get("ppg_change", float("nan"))):
            change = row['ppg_change']
            trend = "📈 Improving" if change > 0 else "📉 Declining"
            st.success(f"{trend}: {abs(change):.1f} PPG change over career")

# =============================================================================
# TAB 2: EDA
# =============================================================================

with tab_eda:
    st.header("📊 Exploratory Data Analysis")

    # Summary statistics
    st.subheader("Dataset Overview")

    col1, col2, col3, col4, col5 = st.columns(5)

    with col1:
        st.metric("Total Prospects", stats['total_prospects'])
    with col2:
        st.metric("Avg Age", f"{stats['avg_age']:.1f}")
    with col3:
        st.metric("Avg PPG", f"{stats['avg_ppg']:.1f}")
    with col4:
        st.metric("With NBA Exp", stats['with_nba_exp'])
    with col5:
        st.metric("Leagues", stats['leagues'])

    st.markdown("---")

    # Age distribution
    st.subheader("Age Distribution")
    fig_age = create_age_distribution_plot(f)
    st.plotly_chart(fig_age, use_container_width=True)

    # Performance scatter
    st.subheader("Performance Analysis")
    fig_perf = create_performance_scatter(f)
    st.plotly_chart(fig_perf, use_container_width=True)

    st.markdown("""
    **Insights**:
    - Bubble size represents minutes played (larger = more playing time)
    - Color represents age (darker = older players)
    - Look for high PPG + high efficiency + reasonable age
    """)

    st.markdown("---")

    # League comparison
    if 'league' in f.columns:
        st.subheader("League Comparison")
        fig_league = create_league_comparison(f)
        if fig_league:
            st.plotly_chart(fig_league, use_container_width=True)

    # Shooting analysis
    st.subheader("Shooting Efficiency Distribution")
    fig_shoot = create_shooting_analysis(f)
    st.plotly_chart(fig_shoot, use_container_width=True)

    st.markdown("---")

    # Improvement analysis
    if 'ppg_change' in f.columns:
        st.subheader("Player Development Trends")
        fig_improve = create_improvement_analysis(f)
        if fig_improve:
            st.plotly_chart(fig_improve, use_container_width=True)

            improving_count = (f['ppg_change'] > 3).sum()
            st.info(f"📈 **{improving_count}** players show significant improvement (> 3 PPG growth)")

    # Success probability
    if 'nba_success_prob' in f.columns:
        st.subheader("NBA Success Probability Distribution")
        fig_prob = create_success_probability_distribution(f)
        if fig_prob:
            st.plotly_chart(fig_prob, use_container_width=True)

    st.markdown("---")

    # Correlation heatmap
    st.subheader("Statistical Correlations")
    fig_corr = create_correlation_heatmap(f)
    st.plotly_chart(fig_corr, use_container_width=True)

    st.markdown("""
    **Key Relationships**:
    - **High correlation** (red): Variables that move together
    - **Low/negative correlation** (blue): Independent or inverse relationships
    - Scout score components show expected relationships with performance metrics
    """)

# =============================================================================
# TAB 3: MODEL DIAGNOSTICS
# =============================================================================

with tab_model:
    st.header("🤖 Model Diagnostics & Performance")

    if metrics:
        st.subheader("Model Performance Summary")

        col1, col2 = st.columns(2)

        with col1:
            st.markdown("### Classification Metrics")
            metrics_df = pd.DataFrame([
                {"Metric": "ROC-AUC (Test)", "Value": f"{metrics.get('test_auc', 0):.3f}"},
                {"Metric": "PR-AUC (Test)", "Value": f"{metrics.get('test_pr_auc', 0):.3f}"},
                {"Metric": "CV ROC-AUC (Median)", "Value": f"{metrics.get('cv_auc_median', 0):.3f}"},
                {"Metric": "CV PR-AUC (Median)", "Value": f"{metrics.get('cv_pr_auc_median', 0):.3f}"},
            ])
            st.dataframe(metrics_df, use_container_width=True, hide_index=True)

        with col2:
            st.markdown("### Calibration & Precision")
            calib_df = pd.DataFrame([
                {"Metric": "Brier Score (Calibrated)", "Value": f"{metrics.get('brier_score_calibrated', 0):.3f}"},
                {"Metric": "Brier Score (Uncalibrated)", "Value": f"{metrics.get('brier_score_uncalibrated', 0):.3f}"},
                {"Metric": "Brier Improvement", "Value": f"{metrics.get('brier_improvement', 0):.3f}"},
                {"Metric": "Precision@10", "Value": f"{metrics.get('precision_at_10', 0):.1%}"},
            ])
            st.dataframe(calib_df, use_container_width=True, hide_index=True)

        st.markdown("---")

        st.info("""
        **Model Interpretation**:
        - **PR-AUC** is the primary metric due to class imbalance (few NBA successes)
        - **Brier Score** measures calibration quality (lower is better)
        - **Precision@K** shows accuracy in top-K predictions (most relevant for scouting)
        - **Cross-validation** results show model stability
        """)

    st.markdown("---")

    # Feature importance
    if not fi.empty:
        st.subheader("Feature Importance Analysis")

        fig_fi = create_feature_importance_plot(fi)
        if fig_fi:
            st.plotly_chart(fig_fi, use_container_width=True)

        st.markdown("### Detailed Feature Rankings")
        fi_display = fi.sort_values('importance_perm', ascending=False).head(20)
        st.dataframe(
            fi_display.style.background_gradient(subset=['importance_perm'], cmap='Blues'),
            use_container_width=True
        )

        st.markdown("""
        **Feature Importance Interpretation**:
        - **Permutation importance** measures impact on PR-AUC when feature is shuffled
        - Higher values = more critical for predictions
        - Error bars show variability across permutations
        - Top features align with basketball domain knowledge (PPG, efficiency, shooting)
        """)

    st.markdown("---")

    # Diagnostic plots
    st.subheader("Visual Diagnostics")

    col1, col2 = st.columns(2)

    with col1:
        try:
            st.image("ml_diagnostic_plots.png", caption="Calibration, ROC, and PR Curves")
        except Exception:
            st.warning("Diagnostic plots image not found")

    with col2:
        try:
            st.image("eda_visualizations_.png", caption="Comprehensive EDA (Static)")
        except Exception:
            st.warning("EDA visualization image not found")

    st.markdown("---")

    # Scout score breakdown
    st.subheader("Scout Score Component Analysis")
    fig_scout = create_scout_score_breakdown(f)
    st.plotly_chart(fig_scout, use_container_width=True)

# =============================================================================
# TAB 4: PROCESS & METHODOLOGY
# =============================================================================

with tab_process:
    st.header("📋 Process & Methodology")

    st.markdown("""
    ## Data Pipeline Overview

    This analysis follows a comprehensive, modular pipeline for international basketball scouting:

    ### 1️⃣ Data Loading & Validation
    - **Sources**: NBA and International (EuroLeague, ACB, VTB, etc.) statistics (2010-2021)
    - **Validation**: Automated quality checks for missing values, outliers, and inconsistencies
    - **Normalization**: Standardized metrics across leagues (TS%, efficiency, per-game stats)

    ### 2️⃣ Feature Engineering
    - **Per-game statistics**: PPG, APG, RPG, SPG, BPG
    - **Advanced metrics**: True Shooting %, Efficiency Rating, Usage %
    - **Trajectory features**: PPG trends, improvement over time
    - **Contextual features**: Age, league quality, playing time

    ### 3️⃣ Machine Learning Model
    - **Algorithm**: Calibrated Gradient Boosting Classifier
    - **Target**: NBA success (MPG ≥ 15 AND meaningful statistical contribution)
    - **Training**: Players with both international and NBA experience
    - **Calibration**: Sigmoid calibration for probability reliability
    - **Validation**: 5-fold cross-validation + hold-out test set

    ### 4️⃣ Scouting Score Calculation

    The scout score combines multiple components:

    ```
    scout_score = performance_score × age_bonus × improvement_bonus ×
                  fit_multiplier × ml_multiplier
    ```

    Where:
    - **Performance score**: Weighted combination of PPG, efficiency, TS%, APG, RPG
    - **Age bonus**: 1.3x for <24, 1.2x for 24-26, 1.1x for 26-28, 1.0x for 28+
    - **Improvement bonus**: Based on PPG trajectory (1.2x for strong growth)
    - **Fit multiplier**: Team-specific needs (shooting, defense, playmaking)
    - **ML multiplier**: Calibrated NBA success probability adjustment

    ### 5️⃣ Quality Assurance
    - **Data quality log**: All issues tracked and documented
    - **Sanity checks**: SQL queries verify data integrity
    - **Diagnostic visualizations**: Monitor distributions and outliers
    - **Model diagnostics**: Calibration curves, confusion matrices

    ## Key Design Decisions

    ### Why PR-AUC over ROC-AUC?
    - Dataset is **imbalanced** (few NBA successes vs. many non-successes)
    - PR-AUC better captures performance on minority class
    - ROC-AUC can be misleadingly optimistic with imbalance

    ### Why Calibrated Probabilities?
    - Raw model outputs may be overconfident or underconfident
    - Calibration ensures probabilities are **trustworthy** (70% means 70%)
    - Brier score measures calibration quality

    ### Why Multiple Scout Score Components?
    - **Holistic evaluation**: Not just current performance
    - **Context matters**: Age, improvement trajectory, team fit
    - **Risk mitigation**: ML probability tempered by traditional scouting

    ## Data Quality & Limitations

    ### Quality Controls Implemented:
    ✅ Duplicate name detection and resolution
    ✅ Out-of-range value flagging (shooting %, games, minutes)
    ✅ Scale normalization (0-100 vs 0-1 percentages)
    ✅ Orphaned record removal (stats without player demographics)
    ✅ Missing value documentation

    ### Known Limitations:
    ⚠️ Data anonymized per assignment (identities hidden)
    ⚠️ No post-2021 validation possible
    ⚠️ League quality differences not fully captured
    ⚠️ Small sample size for NBA success (class imbalance)
    ⚠️ Temporal data may have leakage (players with concurrent NBA/Intl careers)

    ## Assignment-Specific Notes

    - All player names are **anonymized** to demonstrate process, not outcomes
    - Focus is on **methodology** and **reproducibility**
    - Code is **modular** and **well-documented** for easy review
    - Outputs include **diagnostics** for transparency
    """)

    st.markdown("---")

    st.subheader("📁 Output Files Generated")

    output_files = pd.DataFrame([
        {"File": "final_scouting_report.csv", "Description": "Top 30 prospects with all metrics"},
        {"File": "ml_metrics.json", "Description": "Model performance metrics"},
        {"File": "feature_importance.csv", "Description": "Feature rankings"},
        {"File": "data_quality_report.txt", "Description": "Data quality issues log"},
        {"File": "eda_visualizations_.png", "Description": "9-panel EDA visualization"},
        {"File": "ml_diagnostic_plots.png", "Description": "Calibration, ROC, PR curves"},
        {"File": "kings_scouting.db", "Description": "SQLite database with all data"},
    ])

    st.dataframe(output_files, use_container_width=True, hide_index=True)

# =============================================================================
# TAB 5: LIMITATIONS
# =============================================================================

with tab_limits:
    st.header("⚠️ Limitations & Considerations")

    st.markdown("""
    ## Data Limitations

    ### 1. Anonymization
    - Player identities are **anonymized** per assignment requirements
    - Prevents real-world validation of recommendations
    - Focus is on demonstrating **process quality**, not actual outcomes

    ### 2. Temporal Constraints
    - Data ends in **2021**
    - Cannot validate predictions against post-2021 NBA performance
    - Some players may have since proven/disproven model predictions

    ### 3. Sample Size
    - **Class imbalance**: Few NBA successes relative to international players
    - Limits model confidence, especially for rare player archetypes
    - PR-AUC and calibration help mitigate but don't eliminate this issue

    ### 4. League Quality Adjustment
    - Different international leagues have varying competition levels
    - Model implicitly captures this through historical data
    - Could be  with explicit league strength ratings

    ## Model Limitations

    ### 1. Feature Coverage
    - **Missing factors**: Injury history, personality, work ethic, team culture fit
    - **Intangibles**: Leadership, clutch performance, defensive IQ
    - **Context**: Coaching, system fit, role availability

    ### 2. Temporal Leakage Risk
    - Some players played internationally **while** in NBA (e.g., lockout, buyouts)
    - Feature extraction attempts to use only pre-NBA international data
    - Diagnostic checks flag potential leakage cases

    ### 3. Generalization
    - Model trained on **historical** NBA success criteria
    - Game evolution (3-point emphasis, pace, positionless basketball) may shift success factors
    - Periodic retraining recommended

    ## Operational Considerations

    ### 1. Probability Interpretation
    - **70% success probability ≠ 70% certainty**
    - Reflects model's confidence based on historical similar players
    - Should inform, not replace, human scouting judgment

    ### 2. Scout Score Trade-offs
    - High score favors **young, improving, efficient** players
    - May undervalue **veterans** with specialized skills
    - May overvalue **high-volume scorers** in weak leagues

    ### 3. Recommendation Usage
    - Use as **screening tool** to prioritize in-depth scouting
    - Combine with **video analysis**, **interviews**, **medical evaluations**
    - Consider **organizational needs** and **roster construction**

    ## Ethical Considerations

    ### 1. Bias & Fairness
    - Historical NBA success criteria may embed systemic biases
    - International league representation may vary by region
    - Model should be audited for demographic fairness

    ### 2. Transparency
    - All code is **open** and **documented**
    - Feature importance and diagnostics provided
    - Anonymization protects player privacy

    ### 3. Human Oversight
    - ML should **augment**, not replace, human scouts
    - Final decisions require contextual judgment
    - Model outputs are **recommendations**, not mandates

    ## Future Improvements

    ### Potential Enhancements:
    - ✨ Incorporate **play-by-play** data for deeper insights
    - ✨ Add **video analysis** features (shot selection, defensive positioning)
    - ✨ Include **combine measurements** (athleticism, length)
    - ✨ Model **different success tiers** (starter vs. rotation vs. end-of-bench)
    - ✨ Implement **Bayesian updating** as more data arrives
    - ✨ Build **similarity search** to find NBA comparables

    ---

    ## Contact & Feedback

    This analysis demonstrates a **structured, reproducible pipeline** for sports analytics.
    For questions about methodology or implementation, refer to the codebase documentation.
    """)

# =============================================================================
# FOOTER
# =============================================================================

st.markdown("---")
st.markdown("""
<div style='text-align: center; color: #666; padding: 20px;'>
    <p><strong>Sacramento Kings International Scouting Analysis</strong></p>
    <p>Assignment Demonstration | Data: 2010-2021 | Player Names Anonymized</p>
    <p>Built with Streamlit, Plotly, scikit-learn | Modular, Reproducible Pipeline</p>
</div>
""", unsafe_allow_html=True)


Overwriting app.py
